# 🛰️ Jalna End-to-End Remote Sensing Pipeline

## Overview
This notebook combines four previously separate notebooks into a **single end-to-end pipeline**:

| Stage | Source Notebook | Description |
|-------|----------------|-------------|
| **1. Download** | `preserve_download.ipynb` | Downloads S1/S2 Sentinel imagery from Copernicus OpenEO into a `pairs/` folder structure |
| **2. Cloud-Fill** | `All_Mills.ipynb` | Runs a 5-tier SAR-optical fusion Random Forest model to produce cloud-free NDRE/NDWI GeoTIFFs |
| **3. KPI Processing** | `Processing_Code_new.ipynb` | Clips rasters to parcel boundaries and computes Health, Water Stress, Harvest, Fertilizer and Weed KPIs |
| **4. S3 Transfer** | `S3_transfer.ipynb` | Backs up raw pairs data and all outputs to the S3 bucket |

## How to Run
1. **Edit only Cell 0** — the master configuration block.
2. Set the `AOI_GEOJSON`, `fecha_inicio`, `fecha_fin`, and any S3/path settings.
3. Run all cells top-to-bottom (Kernel → Restart & Run All).

## Pipeline Toggle Flags (in Cell 0)
- `RUN_DOWNLOAD` — set `False` if imagery is already downloaded
- `RUN_CLOUDFILL` — set `False` to skip the RF cloud-removal step
- `RUN_KPI` — set `False` to skip KPI parquet generation
- `RUN_S3_UPLOAD_RAW` — upload the raw `pairs/` folder to S3
- `RUN_S3_UPLOAD_OUTPUTS` — upload cloud-free TIFFs and KPI parquets to S3


## ⚙️ Cell 0 — Master Configuration
> **This is the ONLY cell you need to edit for a new run.**

In [ ]:
# ============================================================================
# CELL 0: MASTER CONFIGURATION
# ============================================================================
# Edit the variables in this cell to configure the pipeline.
# Everything downstream reads from these values — no other cell needs editing.
# ============================================================================
import os
import sys
from pathlib import Path
from datetime import datetime

# ── 1. INPUT DATA ──────────────────────────────────────────────────────────
# Path to the GeoJSON file defining the Area of Interest (AOI).
# Used for: downloading imagery, cloud-fill spatial extent, KPI parcel clipping.
AOI_GEOJSON = Path("/home/sagemaker-user/JALNA/Khushl/data_download/Jalana_AOI_extended_east.geojson")

# Path to the GeoJSON file with the individual field/parcel polygons used for KPI extraction.
# Must contain columns: parcel_id, sr_no, plot_type (geometry in any CRS — reprojected automatically)
PARCELS_GEOJSON = Path("/home/sagemaker-user/Jalna_Seminar_Data/Second Version 27-04-2026/10_parcels_clean.geojson")

# ── 2. DATE RANGE ──────────────────────────────────────────────────────────
# The 5-day window in which the "inference date" (most recent clear S2 image)
# will be searched. The downloader selects the latest spatially-complete date
# within this window.
fecha_inicio = datetime(2026, 4, 20)   # Start of inference window (inclusive)
fecha_fin    = datetime(2026, 5, 6)    # End   of inference window (inclusive)

# ── 3. LOCAL DIRECTORIES ───────────────────────────────────────────────────
# Base working directory for this AOI / mill.
BASE_DIR = Path("/home/sagemaker-user/JALNA/Khushl")

# Sub-directories (auto-created if they do not exist):
PAIRS_DIR          = BASE_DIR / "pairs"          # S1+S2 image pairs (download output)
CLOUDFILL_DIR      = BASE_DIR / "cloudfill_out"  # Cloud-free TIFFs (cloud-fill output)
KPI_OUTPUT_DIR     = BASE_DIR / "kpi_outputs"    # KPI parquet files (KPI output)
SOWING_DATES_PATH  = Path("/home/sagemaker-user/Jalna_Seminar_Data/Second Version 27-04-2026/sowing_dates.json")

# ── 4. S3 CONFIGURATION ────────────────────────────────────────────────────
S3_BUCKET = "carrier-pdfs"

# S3 prefix for raw pairs backup  → s3://<bucket>/<S3_RAW_PREFIX>/
S3_RAW_PREFIX = f"JALNA/raw_pairs/{fecha_fin.strftime('%Y-%m-%d')}"

# S3 prefix for processed outputs → s3://<bucket>/<S3_OUTPUT_PREFIX>/
S3_OUTPUT_PREFIX = f"JALNA/outputs/{fecha_fin.strftime('%Y-%m-%d')}"

# ── 5. DOWNLOAD CONFIGURATION ──────────────────────────────────────────────
LOOKBACK_DAYS           = 350   # How many days back to search for previous S2 images
PREVIOUS_IMAGES_COUNT   = 10    # Number of previous clear S2 dates to keep in pairs/
S1_DATE_BUFFER          = 60    # Days of S1 GRD data for the median composite

# ── 6. CLOUD-FILL CONFIGURATION ────────────────────────────────────────────
# Site/mill identifier used in output filenames.
# For Jalna use "JL". Add more entries to SITE_CONFIGS in Cell 3 if needed.
SITE = "JL"

# Set to True to also save 100% synthetic (model-only) full S2 image
SAVE_SYNTHETIC_FULL = False

# Set to True to run post-inference validation metrics (R², MAE, RMSE, SSIM)
RUN_VALIDATION = False

# Validation reference date (only used when RUN_VALIDATION = True)
GTE3_REFERENCE_DATE = "2025-05-03"

# ── 7. KPI CONFIGURATION ───────────────────────────────────────────────────
# Default planting date and cane cycle length (used if sowing_dates.json is absent
# or a farmer's entry is missing)
DEFAULT_PLANTING_DATE = "2025-12-01"
DEFAULT_CYCLE_DAYS    = 450

# Toggle to also generate weed maps (GeoTIFFs). Set False to skip and save time.
GENERATE_WEED_MAPS = False

# ── 8. PIPELINE STAGE TOGGLES ──────────────────────────────────────────────
# Set any stage to False to skip it (useful for re-running from a mid-point).
RUN_DOWNLOAD          = True   # Stage 1: Download S1/S2 from OpenEO
RUN_CLOUDFILL         = True   # Stage 2: Cloud-fill Random Forest pipeline
RUN_KPI               = True   # Stage 3: KPI parquet generation
RUN_S3_UPLOAD_RAW     = True   # Stage 4a: Upload raw pairs/ to S3
RUN_S3_UPLOAD_OUTPUTS = True   # Stage 4b: Upload cloudfill + KPI outputs to S3

# ── 9. OpenEO BACKEND ──────────────────────────────────────────────────────
OPENEO_BACKEND = "https://openeo.dataspace.copernicus.eu"

# ── DISPLAY SUMMARY ────────────────────────────────────────────────────────
print("=" * 70)
print("  JALNA END-TO-END PIPELINE — CONFIGURATION")
print("=" * 70)
print(f"  AOI GeoJSON        : {AOI_GEOJSON}")
print(f"  Parcels GeoJSON    : {PARCELS_GEOJSON}")
print(f"  Inference window   : {fecha_inicio.date()} → {fecha_fin.date()}")
print(f"  Base directory     : {BASE_DIR}")
print(f"  Pairs directory    : {PAIRS_DIR}")
print(f"  Cloud-fill output  : {CLOUDFILL_DIR}")
print(f"  KPI output         : {KPI_OUTPUT_DIR}")
print(f"  S3 bucket          : {S3_BUCKET}")
print(f"  S3 raw prefix      : {S3_RAW_PREFIX}")
print(f"  S3 output prefix   : {S3_OUTPUT_PREFIX}")
print(f"\n  STAGES TO RUN:")
print(f"    Download          : {'✅ YES' if RUN_DOWNLOAD else '⏭️  SKIP'}")
print(f"    Cloud-fill        : {'✅ YES' if RUN_CLOUDFILL else '⏭️  SKIP'}")
print(f"    KPI Processing    : {'✅ YES' if RUN_KPI else '⏭️  SKIP'}")
print(f"    S3 Upload (raw)   : {'✅ YES' if RUN_S3_UPLOAD_RAW else '⏭️  SKIP'}")
print(f"    S3 Upload (out)   : {'✅ YES' if RUN_S3_UPLOAD_OUTPUTS else '⏭️  SKIP'}")
print("=" * 70)


## 📦 Cell 1 — Imports & Environment Setup

In [ ]:
# ============================================================================
# CELL 1: IMPORTS & ENVIRONMENT SETUP
# ============================================================================
# All third-party imports needed by every downstream stage.
# If a library is missing, install it via:
#   !pip install <library_name>
# ============================================================================

import os
import gc
import re
import sys
import json
import time
import shutil
import logging
import tempfile
import warnings
import traceback
from collections import defaultdict, OrderedDict
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import boto3

from shapely.geometry import Point, Polygon, box, MultiPolygon, shape
from shapely.ops import unary_union
from shapely import wkt

# ── Optional heavy deps (checked at runtime) ───────────────────────────────
try:
    import mgrs
    HAS_MGRS = True
except ImportError:
    HAS_MGRS = False
    print("⚠️  mgrs not installed — MGRS tile listing unavailable (non-critical)")

try:
    import openeo
    HAS_OPENEO = True
except ImportError:
    HAS_OPENEO = False
    print("⚠️  openeo not installed — download stage will fail")

try:
    import rasterio
    from rasterio.warp import calculate_default_transform, reproject, Resampling
    from rasterio.mask import mask as rasterio_mask
    from rasterio.crs import CRS
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    print("⚠️  rasterio not installed")

try:
    from scipy import ndimage
    from scipy.ndimage import uniform_filter
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from tqdm import tqdm
    HAS_ML = True
except ImportError:
    HAS_ML = False
    print("⚠️  scipy / sklearn / tqdm not installed — cloud-fill stage will fail")

import matplotlib.pyplot as plt
import matplotlib

warnings.filterwarnings("ignore")

# ── Create all output directories from Cell 0 config ──────────────────────
for d in [PAIRS_DIR, CLOUDFILL_DIR, KPI_OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ All imports loaded.")
print(f"   rasterio  : {'✅' if HAS_RASTERIO else '❌'}")
print(f"   openeo    : {'✅' if HAS_OPENEO else '❌'}")
print(f"   sklearn   : {'✅' if HAS_ML else '❌'}")
print(f"   mgrs      : {'✅' if HAS_MGRS else '❌ (optional)'}")
print()
print("📁 Output directories ready:")
print(f"   pairs/         → {PAIRS_DIR}")
print(f"   cloudfill_out/ → {CLOUDFILL_DIR}")
print(f"   kpi_outputs/   → {KPI_OUTPUT_DIR}")


## 🛰️ Cell 2 — S1/S2 Download Pipeline

Downloads Sentinel-1 and Sentinel-2 imagery for the configured AOI and date range
using the Copernicus OpenEO API.

**Key behaviours:**
- Scans the last `LOOKBACK_DAYS` days for spatially-complete S2 L2A images (≥95% AOI coverage)
- Selects the latest complete date inside the 5-day inference window as `INFERENCE_DATE`
- Keeps the `PREVIOUS_IMAGES_COUNT` most recent complete dates before the inference date
- Downloads S2 as a 15-band GeoTIFF (all bands at 10 m) via an OpenEO batch job
- Downloads S1 as a 60-day median composite GRD (VV + VH, no sar_backscatter)
- Organises files into `pairs/inference_YYYY-MM-DD/` and `pairs/prevNN_YYYY-MM-DD/` folders
- On re-run: promotes the old inference folder to `prev01`, shifts everything up — never re-downloads existing valid files

**Outputs (exported to Python scope for downstream cells):**
- `INFERENCE_DATE` — string, e.g. `"2026-05-03"`
- `PREVIOUS_DATES` — list of date strings
- `DOWNLOAD_PAIRS` — list of dicts with s2_date, s1_date, folder info
- `S2_FILE_PATHS`, `S1_FILE_PATHS` — dicts mapping date → Path


In [ ]:
# ============================================================================
# CELL 2: S1/S2 DOWNLOAD PIPELINE
# ============================================================================
# Source: preserve_download.ipynb
#
# All configuration is read from Cell 0.  The only runtime input is your
# OpenEO OIDC credentials (browser prompt on first run; cached afterwards).
# ============================================================================

if not RUN_DOWNLOAD:
    print("⏭️  RUN_DOWNLOAD = False — skipping download stage.")
    print("   Assuming pairs/ folder already contains valid S1/S2 files.")
    # ── Minimal stub so downstream cells still find the variables ──────────
    # Scan existing pairs/ folder and reconstruct the key variables.
    import re as _re

    def _scan_pairs(pairs_dir):
        result = {"inference": None, "previous": [], "all_s2_dates": set(), "all_s1_dates": set(), "all_dates": {}}
        if not pairs_dir.exists():
            return result
        date_pat = _re.compile(r"(\d{4}-\d{2}-\d{2})")
        for folder in sorted(pairs_dir.iterdir()):
            if not folder.is_dir():
                continue
            m = date_pat.search(folder.name)
            if not m:
                continue
            d = m.group(1)
            result["all_dates"][d] = folder
            if "inference" in folder.name:
                result["inference"] = {"date": d, "folder": folder}
            elif "prev" in folder.name:
                idx_m = _re.match(r"prev(\d+)_", folder.name)
                result["previous"].append({"date": d, "folder": folder, "index": int(idx_m.group(1)) if idx_m else 99})
        result["previous"].sort(key=lambda x: x["index"])
        return result

    _existing = _scan_pairs(PAIRS_DIR)
    if _existing["inference"]:
        INFERENCE_DATE = _existing["inference"]["date"]
        PREVIOUS_DATES = [p["date"] for p in _existing["previous"]]
        print(f"   Detected inference date : {INFERENCE_DATE}")
        print(f"   Previous dates ({len(PREVIOUS_DATES)})    : {PREVIOUS_DATES}")
    else:
        raise RuntimeError(
            f"RUN_DOWNLOAD=False but no inference folder found in {PAIRS_DIR}.\n"
            "Either set RUN_DOWNLOAD=True or ensure pairs/ contains valid folders."
        )
    # Rebuild stub dicts — cloud-fill will find actual files by scanning folders
    DOWNLOAD_PAIRS = [{"s2_date": INFERENCE_DATE, "s1_date": None, "role": "inference", "prev_index": 0}]
    for i, d in enumerate(PREVIOUS_DATES, 1):
        DOWNLOAD_PAIRS.append({"s2_date": d, "s1_date": None, "role": "previous", "prev_index": i})
    S2_FILE_PATHS = {}
    S1_FILE_PATHS = {}

else:
    # =========================================================================
    # ── FULL DOWNLOAD PIPELINE ───────────────────────────────────────────────
    # =========================================================================

    # ── OpenEO connection factory ────────────────────────────────────────────
    def get_openeo_connection(force_refresh=False):
        """
        Returns an authenticated OpenEO connection.
        On first call: opens a browser for OIDC login.
        On subsequent calls in the same session: reuses cached token.
        """
        conn = openeo.connect(OPENEO_BACKEND)
        conn.authenticate_oidc()
        return conn

    # ── Satellite config (from Cell 0 + fixed constants) ────────────────────
    TARGET_CRS         = "EPSG:4326"
    MARGIN_DEGREES     = 0.001
    MIN_SPATIAL_COV    = 95.0    # % AOI coverage required for a "complete" S2 date
    MAX_NODATA_PCT     = 10.0    # Max % nodata allowed in preview validation
    PREVIEW_RESOLUTION = 100     # metres, for fast nodata preview check

    S2_COLLECTION    = "SENTINEL2_L2A"
    S2_BANDS         = ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B11","B12","WVP","AOT","SCL"]
    S2_EXPECTED_BANDS = len(S2_BANDS)   # 15
    TARGET_RESOLUTION = 10

    S1_COLLECTION    = "SENTINEL1_GRD"
    S1_BANDS         = ["VV","VH"]
    S1_EXPECTED_BANDS = len(S1_BANDS)  # 2

    MAX_RETRIES    = 3
    POLL_INTERVAL  = 30
    JOB_TIMEOUT    = 7200
    CHUNK_SIZE_DL  = 8 * 1024 * 1024
    TIMEOUT_DL     = 300
    RETRY_DELAY    = 10

    # ── Raster band-count helpers ─────────────────────────────────────────────
    def get_band_count(filepath):
        if not HAS_RASTERIO or not filepath or not Path(filepath).exists():
            return 0
        try:
            with rasterio.open(filepath) as src:
                return src.count
        except Exception:
            return 0

    def is_valid_s1_file(fp):
        fp = Path(fp) if fp else None
        if not fp or not fp.exists() or fp.stat().st_size < 1000:
            return False
        return get_band_count(fp) == S1_EXPECTED_BANDS

    def is_valid_s2_file(fp):
        fp = Path(fp) if fp else None
        if not fp or not fp.exists() or fp.stat().st_size < 1000:
            return False
        return get_band_count(fp) == S2_EXPECTED_BANDS

    # ── Folder-name parsers ──────────────────────────────────────────────────
    def parse_pair_folder_name(folder_name):
        m = re.match(r"^inference_(\d{4}-\d{2}-\d{2})$", folder_name)
        if m:
            return {"role":"inference","index":None,"date":m.group(1)}
        m = re.match(r"^prev(\d{2})_(\d{4}-\d{2}-\d{2})$", folder_name)
        if m:
            return {"role":"previous","index":int(m.group(1)),"date":m.group(2)}
        return None

    def find_s2_file_in_folder(folder):
        folder = Path(folder) if folder else None
        if not folder or not folder.exists():
            return None
        for f in sorted(folder.glob("S2_*.tif"), key=lambda x: x.stat().st_mtime, reverse=True):
            if is_valid_s2_file(f): return f
        for f in sorted(folder.glob("openEO_*.tif"), key=lambda x: x.stat().st_mtime, reverse=True):
            if is_valid_s2_file(f): return f
        return None

    def find_s1_file_in_folder(folder):
        folder = Path(folder) if folder else None
        if not folder or not folder.exists():
            return None
        for f in list(folder.glob("s1_*.tif")) + list(folder.glob("S1_*.tif")):
            if is_valid_s1_file(f): return f
        return None

    # ── Metadata tracker class ─────────────────────────────────────────────
    class DateMetadata:
        """Lightweight container tracking download status per date."""
        def __init__(self, date_str, satellite):
            self.date = date_str
            self.satellite = satellite
            self.cloud_cover_pct   = None
            self.nodata_pct        = None
            self.spatial_coverage_pct = None
            self.tiles             = []
            self.is_complete       = None
            self.is_inference      = False
            self.is_broken_allowed = False
            self.download_path     = None
            self.file_size_mb      = None
            self.valid_pixel_pct   = None
            self.band_count        = None

    # ── AOI loader ────────────────────────────────────────────────────────
    def load_aoi_geometry():
        """Load, reproject and compute metadata for the AOI GeoJSON."""
        if not AOI_GEOJSON.exists():
            raise FileNotFoundError(f"AOI not found: {AOI_GEOJSON}")
        print(f"📍 Loading AOI: {AOI_GEOJSON}")
        gdf = gpd.read_file(AOI_GEOJSON)
        if gdf.crs is None:
            gdf = gdf.set_crs(TARGET_CRS)
        elif gdf.crs.to_string() != TARGET_CRS:
            gdf = gdf.to_crs(TARGET_CRS)
        geom = gdf.geometry.iloc[0]
        bounds = geom.bounds
        centroid = geom.centroid
        utm_zone = int((centroid.x + 180) / 6) + 1
        utm_epsg = 32600 + utm_zone if centroid.y >= 0 else 32700 + utm_zone
        area_km2 = gdf.to_crs(epsg=utm_epsg).geometry.area.iloc[0] / 1e6
        print(f"✅ AOI loaded | Bounds: W={bounds[0]:.4f} S={bounds[1]:.4f} E={bounds[2]:.4f} N={bounds[3]:.4f}")
        print(f"📏 Area: {area_km2:.2f} km²")
        geojson_geom = json.loads(gdf.to_json())["features"][0]["geometry"]
        return gdf, {"west": bounds[0],"south": bounds[1],"east": bounds[2],"north": bounds[3],"crs": TARGET_CRS}, geojson_geom

    def add_margin(extent, margin=MARGIN_DEGREES):
        return {
            "west":  extent["west"]  - margin,
            "south": extent["south"] - margin,
            "east":  extent["east"]  + margin,
            "north": extent["north"] + margin,
            "crs": "EPSG:4326",
        }

    # ── OData catalogue queries ────────────────────────────────────────────
    _ODATA = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

    def _parse_footprint(fp_str):
        if not fp_str:
            return None
        try:
            fp_str = fp_str.strip()
            if fp_str.startswith("geography'"):
                m = re.search(r"SRID=\d+;(.+)'$", fp_str)
                if m: fp_str = m.group(1)
            if fp_str.upper().startswith(("POLYGON","MULTIPOLYGON")):
                return wkt.loads(fp_str)
            if fp_str.startswith("{"):
                return shape(json.loads(fp_str))
        except Exception:
            pass
        return None

    def query_s2_products(spatial_extent, start, end):
        w,s,e,n = spatial_extent["west"],spatial_extent["south"],spatial_extent["east"],spatial_extent["north"]
        filt = (
            "Collection/Name eq 'SENTINEL-2' and "
            "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and "
            "att/OData.CSC.StringAttribute/Value eq 'S2MSI2A') and "
            f"ContentDate/Start ge {start}T00:00:00.000Z and "
            f"ContentDate/Start le {end}T23:59:59.999Z and "
            f"OData.CSC.Intersects(area=geography'SRID=4326;"
            f"POLYGON(({w} {s},{e} {s},{e} {n},{w} {n},{w} {s}))')"
        )
        print(f"\n📡 Querying S2 L2A: {start} → {end}")
        dates_products = defaultdict(dict)
        try:
            resp = requests.get(_ODATA, params={"$filter": filt,"$top": 1000,
                "$orderby": "ContentDate/Start desc","$expand": "Attributes"}, timeout=120)
            resp.raise_for_status()
            products = resp.json().get("value",[])
            print(f"📦 {len(products)} S2 products found")
            for prod in products:
                name = prod.get("Name","")
                dm = re.search(r"_(\d{8})T\d{6}_", name)
                tm = re.search(r"_T(\d{2}[A-Z]{3})_", name)
                if not dm or not tm: continue
                d = dm.group(1)
                date_str = f"{d[:4]}-{d[4:6]}-{d[6:8]}"
                tile_id = tm.group(1)
                fp = _parse_footprint(prod.get("Footprint","") or prod.get("GeoFootprint",""))
                cloud = None
                for attr in prod.get("Attributes",[]):
                    if "cloudcover" in attr.get("Name","").lower():
                        try: cloud = float(attr.get("Value",0))
                        except: pass
                        break
                dates_products[date_str][tile_id] = {"product_name": name,"product_id": prod.get("Id",""),"footprint": fp,"cloud_cover": cloud}
        except requests.exceptions.RequestException as ex:
            print(f"❌ OData error: {ex}")
        return dict(dates_products)

    def query_s1_products(spatial_extent, start, end):
        w,s,e,n = spatial_extent["west"],spatial_extent["south"],spatial_extent["east"],spatial_extent["north"]
        filt = (
            "Collection/Name eq 'SENTINEL-1' and "
            "Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and "
            "att/OData.CSC.StringAttribute/Value eq 'IW_GRDH_1S') and "
            f"ContentDate/Start ge {start}T00:00:00.000Z and "
            f"ContentDate/Start le {end}T23:59:59.999Z and "
            f"OData.CSC.Intersects(area=geography'SRID=4326;"
            f"POLYGON(({w} {s},{e} {s},{e} {n},{w} {n},{w} {s}))')"
        )
        print(f"\n📡 Querying S1 GRD: {start} → {end}")
        dates_products = defaultdict(list)
        try:
            resp = requests.get(_ODATA, params={"$filter": filt,"$top": 1000,"$orderby": "ContentDate/Start desc"}, timeout=120)
            resp.raise_for_status()
            products = resp.json().get("value",[])
            print(f"📦 {len(products)} S1 products found")
            for prod in products:
                name = prod.get("Name","")
                dm = re.search(r"_(\d{8})T\d{6}_", name)
                if dm:
                    d = dm.group(1)
                    date_str = f"{d[:4]}-{d[4:6]}-{d[6:8]}"
                    dates_products[date_str].append({"product_name": name,"product_id": prod.get("Id","")})
        except requests.exceptions.RequestException as ex:
            print(f"❌ OData error: {ex}")
        return dict(dates_products)

    # ── Spatial coverage check ─────────────────────────────────────────────
    def check_spatial_coverage(date, products, aoi_geom):
        if not products:
            return {"date": date,"spatial_coverage_pct": 0.0,"is_complete": False,"tiles":[],"avg_cloud_cover": None,"reason":"No products"}
        aoi_area = aoi_geom.area
        valid_fps, tiles, clouds, ptc = [], [], [], {}
        for tid, info in products.items():
            fp = info.get("footprint")
            cc = info.get("cloud_cover")
            if fp is not None and fp.intersects(aoi_geom):
                valid_fps.append(fp); tiles.append(tid); ptc[tid] = cc
                if cc is not None: clouds.append(cc)
        cov = 0.0
        if valid_fps:
            try: cov = (unary_union(valid_fps).intersection(aoi_geom).area / aoi_area) * 100
            except: pass
        avg_cloud = np.mean(clouds) if clouds else None
        complete = cov >= MIN_SPATIAL_COV
        return {"date": date,"spatial_coverage_pct": cov,"is_complete": complete,"tiles": tiles,"avg_cloud_cover": avg_cloud,"reason":"OK" if complete else f"Spatial:{cov:.1f}%"}

    # ── Progress helpers ───────────────────────────────────────────────────
    def _fmt_time(seconds):
        if seconds < 60: return f"{seconds:.0f}s"
        elif seconds < 3600: return f"{seconds//60:.0f}m {seconds%60:.0f}s"
        else: return f"{seconds//3600:.0f}h {(seconds%3600)//60:.0f}m"

    def _progress_bar(progress, width=40, status=""):
        filled = int(width * progress)
        bar = "█"*filled + "░"*(width-filled)
        sys.stdout.write(f"\r   [{bar}] {progress*100:5.1f}% {status}")
        sys.stdout.flush()

    def wait_for_job(job, timeout=JOB_TIMEOUT):
        """Poll an OpenEO batch job until it finishes or times out."""
        start = time.time()
        last_status = None
        print(f"\n   ⏳ Job {job.job_id} submitted. Polling every {POLL_INTERVAL}s...")
        while True:
            elapsed = time.time() - start
            if elapsed > timeout:
                print(f"\n   ❌ Timeout after {_fmt_time(elapsed)}")
                return False
            try:
                status = job.status()
                job_info = job.describe()
                progress = job_info.get("progress", 0)
                if isinstance(progress, (int,float)):
                    progress = progress/100 if progress>1 else progress
                else:
                    progress = {"created":0.05,"queued":0.10,"running":0.15,"finished":1.0}.get(status, 0.1)
                if status == "running":
                    progress = max(0.15, min(0.95, elapsed/2400))
                if status != last_status:
                    print(); print(f"   📊 {last_status} → {status}"); last_status = status
                if status == "finished":
                    _progress_bar(1.0, status=f"Elapsed:{_fmt_time(elapsed)}")
                    print(f"\n   ✅ Done in {_fmt_time(elapsed)}"); return True
                elif status in ("error","canceled"):
                    print(f"\n   ❌ Job {status}"); return False
                else:
                    _progress_bar(progress, status=f"Elapsed:{_fmt_time(elapsed)}")
                time.sleep(POLL_INTERVAL)
            except Exception as e:
                print(f"\n   ⚠️ Poll error: {e}")
                time.sleep(POLL_INTERVAL)

    def download_job_results(job, output_dir):
        """Download OpenEO job results to output_dir with retry."""
        for attempt in range(MAX_RETRIES):
            try:
                results = job.get_results()
                results.download_files(output_dir)
                downloaded = list(output_dir.glob("*.tif")) + list(output_dir.glob("*.nc"))
                if downloaded:
                    for f in downloaded:
                        print(f"   ✅ {f.name} ({f.stat().st_size/1024/1024:.1f} MB, {get_band_count(f)} bands)")
                    return True
                print(f"   ⚠️ No files after download attempt {attempt+1}")
            except Exception as e:
                print(f"   ❌ Download error: {e}")
                if attempt < MAX_RETRIES-1:
                    time.sleep(30*(attempt+1))
        return False

    def download_with_resume(url, output_path, description="", session=None):
        """Chunked download with resume support."""
        output_path = Path(output_path)
        temp_path = output_path.with_suffix(".partial")
        resume_pos = temp_path.stat().st_size if temp_path.exists() else 0
        headers = {"Range": f"bytes={resume_pos}-"} if resume_pos > 0 else {}
        mode = "ab" if resume_pos > 0 else "wb"
        try:
            fn = session.get if session else requests.get
            response = fn(url, headers=headers, stream=True, timeout=TIMEOUT_DL)
            response.raise_for_status()
            total_size = resume_pos + int(response.headers.get("content-length", 0))
            downloaded = resume_pos
            last_print = time.time()
            with open(temp_path, mode) as f:
                for chunk in response.iter_content(chunk_size=CHUNK_SIZE_DL):
                    if chunk:
                        f.write(chunk); downloaded += len(chunk)
                        if time.time() - last_print > 2:
                            pct = (downloaded/total_size*100) if total_size else 0
                            print(f"            📊 {downloaded/1024/1024:.1f}/{total_size/1024/1024:.1f} MB ({pct:.1f}%)")
                            last_print = time.time()
            if temp_path.exists():
                temp_path.rename(output_path)
            print(f"            ✅ {description} ({output_path.stat().st_size/1024/1024:.1f} MB)")
            return True
        except (requests.exceptions.ChunkedEncodingError, requests.exceptions.ConnectionError, requests.exceptions.Timeout):
            print(f"            ⚠️ Interrupted"); return False
        except Exception as e:
            print(f"            ❌ Error: {e}")
            if temp_path.exists(): temp_path.unlink()
            raise

    def download_cube_with_resume(conn, cube, output_path, description="", job_options=None):
        """Submit an OpenEO batch job, wait for completion and download via resume."""
        print(f"         Creating batch job: {description}...")
        job = cube.create_job(
            title=f"{description}_{Path(output_path).stem}",
            out_format="GTiff",
            job_options=job_options or {}
        )
        print(f"         Starting {job.job_id}...")
        job.start_and_wait(max_poll_interval=60, connection_retry_interval=30, soft_error_max=10)
        print(f"         Job finished, downloading...")
        results = job.get_results()
        assets = results.get_assets()
        if not assets:
            raise Exception("No assets in job results")
        asset = assets[0]
        for attempt in range(1, MAX_RETRIES+1):
            print(f"         Attempt {attempt}/{MAX_RETRIES}")
            if download_with_resume(asset.href, output_path, description, getattr(conn,"_session",None)):
                if Path(output_path).exists() and Path(output_path).stat().st_size > 0:
                    return True
            if attempt < MAX_RETRIES:
                print(f"         Retrying in {RETRY_DELAY}s..."); time.sleep(RETRY_DELAY)
        raise Exception(f"Download failed after {MAX_RETRIES} attempts")

    # ── S2 download ────────────────────────────────────────────────────────
    def download_s2_to_folder(date, target_folder, spatial_extent, connection, idx=1, total=1):
        """
        Download a Sentinel-2 L2A stack (all 15 bands) for a single date.
        Skips if a valid 15-band file already exists in target_folder.
        """
        target_folder = Path(target_folder)
        target_folder.mkdir(parents=True, exist_ok=True)
        existing = find_s2_file_in_folder(target_folder)
        if existing:
            print(f"   ⏭️  S2 exists: {existing.name} ({get_band_count(existing)} bands)")
            return existing
        print(f"\n{'='*60}")
        print(f"🛰️  S2 Download [{idx}/{total}] — {date}")
        print(f"   Output : {target_folder}")
        try:
            cube = connection.load_collection(
                S2_COLLECTION,
                spatial_extent=spatial_extent,
                temporal_extent=[date, date],
                bands=S2_BANDS
            ).resample_spatial(resolution=TARGET_RESOLUTION)
            job = cube.create_job(
                title=f"S2_L2A_{date}",
                out_format="GTiff",
                job_options={"driver-memory":"4g","executor-memory":"4g"}
            )
            job.start_job()
            print(f"   ✅ Job started: {job.job_id}")
            if not wait_for_job(job):
                return None
            if not download_job_results(job, target_folder):
                return None
            result = find_s2_file_in_folder(target_folder)
            if result:
                print(f"   🎉 S2 complete: {result.name} ({result.stat().st_size/1024/1024:.1f} MB, {get_band_count(result)} bands)")
            return result
        except Exception as e:
            print(f"   ❌ S2 error: {e}"); traceback.print_exc(); return None

    # ── S1 download ────────────────────────────────────────────────────────
    def download_s1_to_folder(date, target_folder, spatial_extent, connection, idx=1, total=1):
        """
        Download a Sentinel-1 RAW GRD 60-day median composite (VV + VH).
        Skips if a valid 2-band file already exists in target_folder.
        Uses descending orbit to match preserve_download.ipynb behaviour.
        """
        target_folder = Path(target_folder)
        target_folder.mkdir(parents=True, exist_ok=True)
        existing = find_s1_file_in_folder(target_folder)
        if existing:
            print(f"   ⏭️  S1 exists: {existing.name} ({get_band_count(existing)} bands)")
            return existing
        date_compact = date.replace("-","")
        s1_dt = datetime.strptime(date,"%Y-%m-%d")
        s1_start = (s1_dt - timedelta(days=S1_DATE_BUFFER)).strftime("%Y-%m-%d")
        final_output = target_folder / f"s1_{date_compact}.tif"
        print(f"\n{'='*60}")
        print(f"📡 S1 Download [{idx}/{total}] — {date}")
        print(f"   Composite window : {s1_start} → {date} ({S1_DATE_BUFFER}d)")
        try:
            s1_cube = connection.load_collection(
                S1_COLLECTION,
                spatial_extent=spatial_extent,
                temporal_extent=[s1_start, date],
                bands=S1_BANDS,
                properties={"sat:orbit_state": lambda x: x == "descending"}
            ).reduce_dimension(dimension="t", reducer="median"
            ).resample_spatial(resolution=TARGET_RESOLUTION)
            download_cube_with_resume(
                connection, s1_cube, final_output, f"S1 GRD {date}",
                job_options={"soft-errors":"true","tile-size":512}
            )
            if final_output.exists():
                bands = get_band_count(final_output)
                if bands != S1_EXPECTED_BANDS:
                    print(f"   ❌ WRONG bands: {bands} (expected {S1_EXPECTED_BANDS})")
                    final_output.unlink(); return None
                print(f"   🎉 S1 complete: {final_output.name} ({final_output.stat().st_size/1024/1024:.1f} MB, {bands} bands)")
                return final_output
            return None
        except Exception as e:
            print(f"   ❌ S1 error: {type(e).__name__}: {e}"); traceback.print_exc(); return None

    # ── Pairs folder management ───────────────────────────────────────────
    def scan_existing_pairs(pairs_dir):
        result = {"inference":None,"previous":[],"all_dates":{},"all_s2_dates":set(),"all_s1_dates":set()}
        if not Path(pairs_dir).exists(): return result
        for folder in sorted(Path(pairs_dir).iterdir()):
            if not folder.is_dir(): continue
            parsed = parse_pair_folder_name(folder.name)
            if not parsed: continue
            date_str = parsed["date"]
            s2_file = find_s2_file_in_folder(folder)
            s1_file = find_s1_file_in_folder(folder)
            entry = {"date": date_str,"folder": folder,"s2_exists": s2_file is not None,
                     "s1_exists": s1_file is not None,"s2_file": s2_file,"s1_file": s1_file,
                     "role": parsed["role"],"index": parsed.get("index")}
            if parsed["role"] == "inference":
                result["inference"] = entry
            else:
                result["previous"].append(entry)
            result["all_dates"][date_str] = folder
            if s2_file: result["all_s2_dates"].add(date_str)
            if s1_file:
                m = re.search(r"(\d{4})-?(\d{2})-?(\d{2})", s1_file.name)
                if m: result["all_s1_dates"].add(f"{m.group(1)}-{m.group(2)}-{m.group(3)}")
        result["previous"].sort(key=lambda x: x.get("index",99))
        return result

    def reorganize_pairs(pairs_dir, new_inference_date, new_previous_dates, existing_info):
        """
        Reorganize the pairs/ folder when a new inference date is needed.
        - Shifts existing previous folders up by 1
        - Demotes old inference to prev01
        - Preserves overflow folders (prev11+)
        """
        print(f"\n🔄 Reorganising pairs for new inference: {new_inference_date}")
        existing_inference = existing_info.get("inference")
        existing_previous  = existing_info.get("previous",[])
        if existing_inference and existing_inference["date"] != new_inference_date:
            old_inf_date   = existing_inference["date"]
            old_inf_folder = existing_inference["folder"]
            max_idx = max((e["index"] for e in existing_previous if e["index"]), default=0)
            for idx in range(max_idx, 0, -1):
                for entry in existing_previous:
                    if entry["index"] == idx and entry["folder"].exists():
                        new_name = Path(pairs_dir) / f"prev{idx+1:02d}_{parse_pair_folder_name(entry['folder'].name)['date']}"
                        if entry["folder"] != new_name:
                            if new_name.exists(): shutil.rmtree(new_name)
                            entry["folder"].rename(new_name)
                            print(f"   📁 {entry['folder'].name} → {new_name.name}")
                        break
            new_prev01 = Path(pairs_dir) / f"prev01_{old_inf_date}"
            if old_inf_folder.exists():
                if new_prev01.exists(): shutil.rmtree(new_prev01)
                old_inf_folder.rename(new_prev01)
                print(f"   📁 {old_inf_folder.name} → {new_prev01.name}")
        # Build target folder map
        target_folders = {new_inference_date: Path(pairs_dir)/f"inference_{new_inference_date}"}
        for i, d in enumerate(new_previous_dates, 1):
            target_folders[d] = Path(pairs_dir) / f"prev{i:02d}_{d}"
        dates_s2 = set(); dates_s1 = set()
        for folder in Path(pairs_dir).iterdir():
            if not folder.is_dir(): continue
            parsed = parse_pair_folder_name(folder.name)
            if not parsed: continue
            fd = parsed["date"]
            if fd in target_folders and folder != target_folders[fd]:
                tgt = target_folders[fd]
                if tgt.exists() and tgt != folder: shutil.rmtree(tgt)
                folder.rename(tgt); folder = tgt
            if fd in target_folders:
                if find_s2_file_in_folder(folder): dates_s2.add(fd)
                if find_s1_file_in_folder(folder): dates_s1.add(fd)
        s2_to_dl = [d for d in [new_inference_date]+new_previous_dates if d not in dates_s2]
        return {"dates_already_have_s2": dates_s2,"dates_already_have_s1": dates_s1,
                "s2_dates_to_download": s2_to_dl,"target_folders": target_folders}

    def find_nearest_s1(s1_dates, target):
        t = datetime.strptime(target,"%Y-%m-%d")
        best, best_diff = None, float("inf")
        for d in s1_dates:
            diff = abs((datetime.strptime(d,"%Y-%m-%d")-t).days)
            if diff < best_diff: best_diff = diff; best = d
        return best, best_diff

    # ── STEP 0: Dates ─────────────────────────────────────────────────────
    print("="*70); print("STEP 0: DATE CONFIGURATION"); print("="*70)
    inf_window = []
    current = fecha_inicio
    while current <= fecha_fin:
        inf_window.append(current)
        current += timedelta(days=1)
    # Pad to at least 5 days (required by original logic)
    while len(inf_window) < 5:
        inf_window.append(inf_window[-1]+timedelta(days=1))
    if len(inf_window) > 5:
        print(f"⚠️  Window > 5 days — clamping to 5 from start")
        inf_window = inf_window[:5]
    inference_window_strs = [d.strftime("%Y-%m-%d") for d in inf_window]
    lookback_start = fecha_fin - timedelta(days=LOOKBACK_DAYS)
    lb_start_str = lookback_start.strftime("%Y-%m-%d")
    lb_end_str   = fecha_fin.strftime("%Y-%m-%d")
    print(f"\n📅 Inference window : {inference_window_strs[0]} → {inference_window_strs[-1]}")
    print(f"🔍 Lookback range   : {lb_start_str} → {lb_end_str}")

    # ── STEP 1: AOI ────────────────────────────────────────────────────────
    print("\n"+"="*70); print("STEP 1: LOAD AOI"); print("="*70)
    gdf, spatial_extent, geojson_geom = load_aoi_geometry()
    aoi_geom = gdf.geometry.iloc[0]
    spatial_extent_with_margin = add_margin(spatial_extent)
    aoi_with_margin = box(spatial_extent_with_margin["west"], spatial_extent_with_margin["south"],
                          spatial_extent_with_margin["east"], spatial_extent_with_margin["north"])

    # ── STEP 2: Scan existing pairs ────────────────────────────────────────
    print("\n"+"="*70); print("STEP 2: SCAN EXISTING PAIRS"); print("="*70)
    existing_info = scan_existing_pairs(PAIRS_DIR)
    print(f"\n📂 Pairs dir: {PAIRS_DIR.absolute()}")
    if existing_info["inference"]:
        ei = existing_info["inference"]
        print(f"   Current inference : {ei['date']} (S2:{'✅' if ei['s2_exists'] else '❌'} S1:{'✅' if ei['s1_exists'] else '❌'})")
    else:
        print("   No existing inference folder")
    print(f"   Previous folders  : {len(existing_info['previous'])}")

    # ── STEP 3: Query S2 ───────────────────────────────────────────────────
    print("\n"+"="*70); print("STEP 3: QUERY S2 PRODUCTS"); print("="*70)
    s2_products = query_s2_products(spatial_extent_with_margin, lb_start_str, lb_end_str)
    if not s2_products:
        raise SystemExit("❌ No S2 products found!")
    print(f"📊 S2 on {len(s2_products)} dates")

    # ── STEP 4: Spatial coverage check ────────────────────────────────────
    print("\n"+"="*70); print("STEP 4: SPATIAL COVERAGE CHECK"); print("="*70)
    spatial_results = {}
    print(f"{'Date':<12} {'Coverage':>10} {'Tiles':<6} {'Cloud%':<10} {'InWindow':<9} {'Status'}")
    print("-"*60)
    for date in sorted(s2_products.keys(), reverse=True):
        r = check_spatial_coverage(date, s2_products[date], aoi_with_margin)
        spatial_results[date] = r
        iw = "✅" if date in inference_window_strs else "  "
        cl = f"{r['avg_cloud_cover']:.1f}%" if r["avg_cloud_cover"] is not None else "N/A"
        st = "✅" if r["is_complete"] else "❌"
        print(f"{date:<12} {r['spatial_coverage_pct']:>8.1f}%  {len(r['tiles']):<6} {cl:<10} {iw:<9} {st} {r['reason']}")
    complete_dates = sorted([d for d,r in spatial_results.items() if r["is_complete"]], reverse=True)
    print(f"\n📊 Complete: {len(complete_dates)} | Incomplete: {len(spatial_results)-len(complete_dates)}")

    # ── STEP 5: Determine inference date ──────────────────────────────────
    print("\n"+"="*70); print("STEP 5: DETERMINE INFERENCE DATE"); print("="*70)
    inf_candidates = [d for d in complete_dates if d in inference_window_strs]
    inference_date = None; inference_is_broken = False
    if inf_candidates:
        inference_date = max(inf_candidates)
        print(f"\n✅ INFERENCE: {inference_date} (latest complete in window)")
    else:
        window_avail = [d for d in inference_window_strs if d in spatial_results]
        if window_avail:
            best = max(window_avail, key=lambda d: spatial_results[d]["spatial_coverage_pct"])
            inference_date = best; inference_is_broken = True
            print(f"\n⚠️  No complete image in window — using BROKEN: {inference_date} ({spatial_results[best]['spatial_coverage_pct']:.1f}% coverage)")
        elif complete_dates:
            inference_date = complete_dates[0]
            print(f"\n⚠️  Using nearest complete before window: {inference_date}")
        else:
            raise SystemExit("❌ No inference date available!")

    # ── STEP 6: Select previous dates ─────────────────────────────────────
    prev_candidates = [d for d in complete_dates if d != inference_date and d <= inference_date]
    previous_dates  = prev_candidates[:PREVIOUS_IMAGES_COUNT]
    print(f"\n📋 Selected {len(previous_dates)} previous dates:")
    for i,d in enumerate(previous_dates,1):
        print(f"   {i}. {d} | Coverage: {spatial_results[d]['spatial_coverage_pct']:.1f}%")

    # ── STEP 7: Query & match S1 ───────────────────────────────────────────
    print("\n"+"="*70); print("STEP 7: QUERY & MATCH S1"); print("="*70)
    s1_start_q = (lookback_start - timedelta(days=12)).strftime("%Y-%m-%d")
    s1_end_q   = (fecha_fin + timedelta(days=12)).strftime("%Y-%m-%d")
    s1_products = query_s1_products(spatial_extent_with_margin, s1_start_q, s1_end_q)
    all_s1_dates = sorted(s1_products.keys(), reverse=True)
    print(f"📊 S1 dates available: {len(all_s1_dates)}")
    download_pairs = []
    DATE_METADATA_S2 = {}; DATE_METADATA_S1 = {}
    all_dl_dates = [inference_date] + previous_dates
    print(f"\n{'Role':<12} {'S2 Date':<12} {'S1 Date':<12} {'Δ Days'}")
    print("-"*50)
    for i, s2d in enumerate(all_dl_dates):
        s1d, s1diff = find_nearest_s1(all_s1_dates, s2d) if all_s1_dates else (None, None)
        role = "inference" if s2d == inference_date else "previous"
        pidx = 0 if role == "inference" else (previous_dates.index(s2d)+1 if s2d in previous_dates else i)
        pair = {"s2_date": s2d,"s1_date": s1d,"s1_diff_days": s1diff,"role": role,"prev_index": pidx}
        download_pairs.append(pair)
        m2 = DateMetadata(s2d,"S2"); m2.is_inference=(role=="inference"); m2.band_count=S2_EXPECTED_BANDS
        if s2d in spatial_results: sr=spatial_results[s2d]; m2.spatial_coverage_pct=sr["spatial_coverage_pct"]; m2.cloud_cover_pct=sr["avg_cloud_cover"]; m2.is_complete=sr["is_complete"]
        DATE_METADATA_S2[s2d] = m2
        if s1d and s1d not in DATE_METADATA_S1:
            m1=DateMetadata(s1d,"S1"); m1.is_inference=(role=="inference"); m1.band_count=S1_EXPECTED_BANDS
            DATE_METADATA_S1[s1d] = m1
        r_label = "INFERENCE" if role=="inference" else f"PREV {pidx:02d}"
        print(f"{r_label:<12} {s2d:<12} {s1d or 'N/A':<12} {s1diff or 'N/A'}")

    # ── STEP 8: Reorganise pairs folder ───────────────────────────────────
    print("\n"+"="*70); print("STEP 8: REORGANISE PAIRS FOLDER"); print("="*70)
    PAIRS_DIR.mkdir(parents=True, exist_ok=True)
    reorg = reorganize_pairs(PAIRS_DIR, inference_date, previous_dates, existing_info)
    target_folders = reorg["target_folders"]
    s1_dates_needed = {p["s1_date"]: p["s2_date"] for p in download_pairs if p["s1_date"]}
    s1_already = set()
    for s1d, s2d in s1_dates_needed.items():
        folder = target_folders.get(s2d)
        if folder and find_s1_file_in_folder(folder): s1_already.add(s1d)
    for folder in PAIRS_DIR.iterdir():
        if folder.is_dir():
            s1f = find_s1_file_in_folder(folder)
            if s1f:
                m = re.search(r"(\d{4})-?(\d{2})-?(\d{2})", s1f.name)
                if m: s1_already.add(f"{m.group(1)}-{m.group(2)}-{m.group(3)}")
    s2_to_download = reorg["s2_dates_to_download"]
    s1_to_download = sorted([d for d in s1_dates_needed if d not in s1_already])
    print(f"\n📋 DOWNLOAD PLAN:")
    print(f"   S2 to download : {len(s2_to_download)} → {s2_to_download}")
    print(f"   S2 already have: {sorted(reorg['dates_already_have_s2'])}")
    print(f"   S1 to download : {len(s1_to_download)} → {s1_to_download}")
    print(f"   S1 already have: {sorted(s1_already)}")

    # ── STEP 9: Execute downloads ──────────────────────────────────────────
    print("\n"+"="*70); print("STEP 9: EXECUTE DOWNLOADS"); print("="*70)
    download_results = {"s2_downloaded":[],"s2_failed":[],"s1_downloaded":[],"s1_failed":[]}
    t_dl_start = time.time()

    if s2_to_download or s1_to_download:
        try:
            # S2 downloads
            if s2_to_download:
                print(f"\n📥 Starting S2 downloads ({len(s2_to_download)} dates)...")
                for i, date in enumerate(s2_to_download, 1):
                    dl_conn = get_openeo_connection()
                    folder = target_folders.get(date)
                    if not folder:
                        print(f"   ⚠️ No target folder for {date}")
                        download_results["s2_failed"].append(date); continue
                    result = download_s2_to_folder(date, folder, spatial_extent_with_margin, dl_conn, i, len(s2_to_download))
                    if result:
                        download_results["s2_downloaded"].append(date)
                        DATE_METADATA_S2[date].download_path = result
                    else:
                        download_results["s2_failed"].append(date)
                    print(f"   📊 S2 progress: {i}/{len(s2_to_download)} | ✅ {len(download_results['s2_downloaded'])} ❌ {len(download_results['s2_failed'])}")

            # S1 downloads
            if s1_to_download:
                print(f"\n📥 Starting S1 downloads ({len(s1_to_download)} dates, {S1_DATE_BUFFER}d composite)...")
                for i, s1_date in enumerate(s1_to_download, 1):
                    dl_conn = get_openeo_connection()
                    target_folder = None
                    for pair in download_pairs:
                        if pair["s1_date"] == s1_date:
                            target_folder = target_folders.get(pair["s2_date"]); break
                    if not target_folder:
                        download_results["s1_failed"].append(s1_date); continue
                    result = download_s1_to_folder(s1_date, target_folder, spatial_extent_with_margin, dl_conn, i, len(s1_to_download))
                    if result and get_band_count(result)==S1_EXPECTED_BANDS:
                        download_results["s1_downloaded"].append(s1_date)
                        DATE_METADATA_S1[s1_date].download_path = result
                    else:
                        download_results["s1_failed"].append(s1_date)
                    print(f"   📊 S1 progress: {i}/{len(s1_to_download)} | ✅ {len(download_results['s1_downloaded'])} ❌ {len(download_results['s1_failed'])}")
        except Exception as e:
            print(f"\n❌ Connection error: {e}"); traceback.print_exc()
    else:
        print("\n✅ All files already downloaded — nothing to do.")

    # ── Copy shared S1 files to all paired folders ─────────────────────────
    print("\n🔄 Copying shared S1 files to paired folders...")
    s1_to_s2_map = {}
    for pair in download_pairs:
        if pair["s1_date"]:
            s1_to_s2_map.setdefault(pair["s1_date"],[]).append(pair["s2_date"])
    copies_made = 0
    for s1d, s2_dates in s1_to_s2_map.items():
        if len(s2_dates) <= 1: continue
        source_file = None
        for s2d in s2_dates:
            f = find_s1_file_in_folder(target_folders.get(s2d))
            if f: source_file = f; break
        if not source_file: continue
        for s2d in s2_dates:
            tgt = target_folders.get(s2d)
            if not tgt or find_s1_file_in_folder(tgt): continue
            tgt.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_file, tgt/source_file.name)
            copies_made += 1
            print(f"   📋 Copied {source_file.name} → {tgt.name}/")
    print(f"   ℹ️  {copies_made} copy/copies made.")

    # ── Build file path dicts ──────────────────────────────────────────────
    S2_FILE_PATHS = {}; S1_FILE_PATHS = {}
    for pair in download_pairs:
        folder = target_folders.get(pair["s2_date"])
        if not folder or not folder.exists(): continue
        s2f = find_s2_file_in_folder(folder)
        if s2f: S2_FILE_PATHS[pair["s2_date"]] = s2f
        if pair["s1_date"]:
            s1f = find_s1_file_in_folder(folder)
            if s1f: S1_FILE_PATHS[pair["s2_date"]] = {"s1_date": pair["s1_date"],"s1_file": s1f}

    # ── Final summary ──────────────────────────────────────────────────────
    INFERENCE_DATE    = inference_date
    INFERENCE_IS_BROKEN = inference_is_broken
    PREVIOUS_DATES    = previous_dates
    DOWNLOAD_PAIRS    = download_pairs
    INFERENCE_WINDOW  = inference_window_strs
    SPATIAL_EXTENT_DL = spatial_extent_with_margin
    TARGET_FOLDERS    = target_folders

    s2_valid = sum(1 for p in download_pairs if find_s2_file_in_folder(target_folders.get(p["s2_date"])))
    s1_valid = sum(1 for p in download_pairs if p["s1_date"] and find_s1_file_in_folder(target_folders.get(p["s2_date"])))
    print(f"""
╔══════════════════════════════════════════════════════════╗
║  📊 DOWNLOAD COMPLETE                                    ║
╠══════════════════════════════════════════════════════════╣
║  Inference Date  : {INFERENCE_DATE:<38} ║
║  Previous Dates  : {len(PREVIOUS_DATES):<38} ║
║  S2 Valid Files  : {f"{s2_valid}/{len(download_pairs)} (15 bands each)":<38} ║
║  S1 Valid Files  : {f"{s1_valid}/{len(download_pairs)} (2 bands each)":<38} ║
║  S2 Failures     : {len(download_results["s2_failed"]):<38} ║
║  S1 Failures     : {len(download_results["s1_failed"]):<38} ║
╚══════════════════════════════════════════════════════════╝
""")
    print(f"   INFERENCE_DATE  = {INFERENCE_DATE}")
    print(f"   PREVIOUS_DATES  = {PREVIOUS_DATES}")
    print(f"   S2_FILE_PATHS   = {len(S2_FILE_PATHS)} files")
    print(f"   S1_FILE_PATHS   = {len(S1_FILE_PATHS)} files")


## ☁️ Cell 3 — Cloud-Fill Pipeline (Random Forest SAR-Optical Fusion)

Trains per-band Random Forest models on clear historical S2 + S1 pixel pairs, then applies
a **5-tier inference strategy** to produce cloud-free NDRE and NDWI GeoTIFFs:

| Tier | Method | Condition |
|------|--------|-----------|
| 0 | Clear (no action) | Already cloud-free |
| 1 | SAR Fusion — High Anchor | Valid S1 + direct clear X-1/X-2 optical |
| 2 | Temporal High | 4-6 clear historical obs (weighted regression) |
| 3 | SAR Fusion — Low Anchor | Valid S1 + temporally-reconstructed anchor |
| 4 | Temporal Low | 1-3 clear historical obs (weighted mean) |
| 5 | Spatial | Nearest-neighbour last resort |

**Outputs saved to `CLOUDFILL_DIR`:**
- `NDRE_<date>_cloudfill.tif` — fused NDRE (clear pixels = real S2, cloudy = model)
- `NDWI_<date>_cloudfill.tif` — fused NDWI
- `cloud_free_<date>.tif` — full 15-band fused S2 stack
- `tier_map_<date>.tif` — pixel-level tier attribution map

**Key variables exported:**
- `CLOUDFILL_TIFF_DIR` — Path to output subfolder (date-stamped)
- `NDRE_CLOUDFILL_PATH`, `NDWI_CLOUDFILL_PATH` — Paths to the KPI-input TIFFs


In [ ]:
# ============================================================================
# CELL 3: CLOUD-FILL PIPELINE
# ============================================================================
# Source: All_Mills.ipynb
#
# Reads from PAIRS_DIR (built by Cell 2) and INFERENCE_DATE.
# Writes outputs to CLOUDFILL_DIR.
# All heavy ML config (RF params, tier thresholds) are set below as constants.
# ============================================================================

if not RUN_CLOUDFILL:
    print("⏭️  RUN_CLOUDFILL = False — skipping cloud-fill stage.")
    # Attempt to find existing cloudfill outputs
    _date_compact = INFERENCE_DATE.replace("-","_")
    _ndre_cand = CLOUDFILL_DIR / _date_compact / "tiffs"
    CLOUDFILL_TIFF_DIR = _ndre_cand
    NDRE_CLOUDFILL_PATH = next(_ndre_cand.glob("*NDRE*cloudfill*.tif"), None) if _ndre_cand.exists() else None
    NDWI_CLOUDFILL_PATH = next(_ndre_cand.glob("*NDWI*cloudfill*.tif"), None) if _ndre_cand.exists() else None
    print(f"   NDRE cloudfill : {NDRE_CLOUDFILL_PATH}")
    print(f"   NDWI cloudfill : {NDWI_CLOUDFILL_PATH}")
else:
    # =========================================================================
    # ── SITE CONFIGURATION ───────────────────────────────────────────────────
    # =========================================================================
    # Add more sites here if needed. Only the entry matching SITE (from Cell 0)
    # is used. pairs_dir and output_dir are injected from Cell 0 paths.
    SITE_CONFIGS_CF = {
        "JL": {
            "site_prefix"                : "JL",
            "group_name"                 : "JALNA_Super_50_Farmer",
            "mill_name"                  : "JALNA",
            "pairs_dir"                  : PAIRS_DIR,          # from Cell 0
            "output_dir"                 : CLOUDFILL_DIR,      # from Cell 0
            "geojson_path"               : AOI_GEOJSON,        # from Cell 0
            "exclude_water_from_training": True,
        },
    }

    if SITE not in SITE_CONFIGS_CF:
        raise ValueError(f"SITE \"{SITE}\" not found in SITE_CONFIGS_CF. Add it above.")

    site_cfg_cf = SITE_CONFIGS_CF[SITE]
    CF_PAIRS_DIR   = Path(site_cfg_cf["pairs_dir"])
    CF_OUTPUT_DIR  = Path(site_cfg_cf["output_dir"])
    SITE_PREFIX    = site_cfg_cf["site_prefix"]
    GROUP_NAME     = site_cfg_cf["group_name"]
    MILL_NAME_CF   = site_cfg_cf["mill_name"]
    GEOJSON_PATH_CF = Path(site_cfg_cf["geojson_path"])

    # The TARGET_DATE for the cloud-fill model is the inference date from Cell 2
    TARGET_DATE_CF = INFERENCE_DATE

    # ── Constants ─────────────────────────────────────────────────────────
    REFLECTIVE_BANDS = ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B11","B12"]
    AUXILIARY_BANDS  = ["WVP","AOT","SCL"]
    ALL_BANDS_ORDERED = REFLECTIVE_BANDS + AUXILIARY_BANDS
    NDRE_B1, NDRE_B2 = "B05","B08"
    NDWI_B1, NDWI_B2 = "B08","B11"

    CLOUD_CLASSES         = (3, 8, 9, 10)
    TRAIN_EXCLUDE_CLASSES = (3, 6, 8, 9, 10)
    NODATA_VAL   = -2147483648
    OUTPUT_NODATA = -32768
    UPPER_BOUND  = 15000

    VV_CLIP_MIN, VV_CLIP_MAX = 0.0001, 1.0
    VH_CLIP_MIN, VH_CLIP_MAX = 0.0001, 0.5

    N_SAR_FEATURES    = 14
    N_ANCHOR_FEATURES = 4
    ANCHOR_SENTINEL   = -1.0

    HIGH_CONF_MIN_SUPPORT = 4
    HIGH_CONF_MAX_SUPPORT = 6
    LOW_CONF_MIN_SUPPORT  = 1
    LOW_CONF_MAX_SUPPORT  = 3
    MAX_LOOKBACK_WINDOW   = 10

    SAMPLE_FRACTION      = 0.5
    MAX_SAMPLES_CF       = 200_000
    RANDOM_STATE_CF      = 42
    N_TRAIN_DATES_CF     = 5
    RF_N_ESTIMATORS_CF   = 100
    RF_MAX_DEPTH_CF      = 15
    RF_MIN_SAMPLES_LEAF_CF = 5
    RF_MAX_FEATURES_CF   = "sqrt"
    RF_N_JOBS_CF         = -1
    CHUNK_SIZE_CF        = 500_000

    TIER_0_CLEAR         = 0
    TIER_1_FUSION_HIGH   = 1
    TIER_2_TEMPORAL_HIGH = 2
    TIER_3_FUSION_LOW    = 3
    TIER_4_TEMPORAL_LOW  = 4
    TIER_5_SPATIAL       = 5

    EFFECTIVE_TRAIN_EXCLUDE = TRAIN_EXCLUDE_CLASSES if site_cfg_cf.get("exclude_water_from_training", True) else CLOUD_CLASSES

    # ── Build output dirs ──────────────────────────────────────────────────
    _date_folder = TARGET_DATE_CF.replace("-","_")
    RUN_DIR_CF   = CF_OUTPUT_DIR / _date_folder
    TIFF_DIR_CF  = RUN_DIR_CF / "tiffs"
    LOG_DIR_CF   = RUN_DIR_CF / "logs"
    PNG_DIR_CF   = RUN_DIR_CF / "pngs"
    CSV_DIR_CF   = RUN_DIR_CF / "csvs"
    for d in [TIFF_DIR_CF, LOG_DIR_CF, PNG_DIR_CF, CSV_DIR_CF]:
        d.mkdir(parents=True, exist_ok=True)

    # ── Logger ─────────────────────────────────────────────────────────────
    _run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    _log_path = LOG_DIR_CF / f"{SITE_PREFIX}_{GROUP_NAME}_{MILL_NAME_CF}_{TARGET_DATE_CF}_run_{_run_ts}.log"
    _cf_logger = logging.getLogger(f"cloudfill_{_run_ts}")
    _cf_logger.setLevel(logging.DEBUG)
    if _cf_logger.handlers: _cf_logger.handlers.clear()
    _fh = logging.FileHandler(_log_path, encoding="utf-8")
    _fh.setLevel(logging.DEBUG)
    _fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
    _cf_logger.addHandler(_fh)

    def cf_log(msg=""):
        print(msg)
        _cf_logger.info(msg)

    cf_log("="*70)
    cf_log(f"CLOUD-FILL PIPELINE | {SITE} | {TARGET_DATE_CF}")
    cf_log(f"  Pairs dir  : {CF_PAIRS_DIR}")
    cf_log(f"  Output dir : {CF_OUTPUT_DIR}")
    cf_log(f"  Log file   : {_log_path}")
    cf_log("="*70)

    # ── Helper functions (identical logic to All_Mills.ipynb) ─────────────
    def _is_sensible(arr):
        return np.isfinite(arr) & (arr != NODATA_VAL) & (arr >= 0) & (arr <= UPPER_BOUND)

    def _get_s2s1_paths(date_str):
        """Find S2 and S1 file paths for a date by scanning CF_PAIRS_DIR."""
        folder = None
        for d in CF_PAIRS_DIR.iterdir():
            if d.is_dir() and date_str in d.name:
                folder = d; break
        if not folder: return None, None
        s2_cands = list(folder.glob("openEO_*.tif")) + list(folder.glob("S2_*.tif"))
        s2_path  = s2_cands[0] if s2_cands else None
        s1_cands = [f for f in list(folder.glob("s1_*.tif")) + list(folder.glob("S1_*.tif")) if "_filled" not in f.name]
        s1_path  = s1_cands[0] if s1_cands else None
        return s2_path, s1_path

    def _detect_bands(s2_path):
        """Map band name → zero-based raster index from S2 TIF metadata."""
        all_band_names = REFLECTIVE_BANDS + AUXILIARY_BANDS
        with rasterio.open(s2_path) as src:
            descriptions = [src.descriptions[i] or f"band_{i+1}" for i in range(src.count)]
        band_idxs = {}
        for i, desc in enumerate(descriptions):
            up = desc.upper().strip()
            for b in all_band_names:
                if b.upper() in up:
                    band_idxs[b] = i
        if "SCL" not in band_idxs:
            band_idxs["SCL"] = len(descriptions)-1
            cf_log("  WARNING: SCL not in band descriptions — assuming last band.")
        return band_idxs

    def _load_s2_bands(date_str, band_idxs, bands_to_load=None):
        s2_path, _ = _get_s2s1_paths(date_str)
        if not s2_path or not s2_path.exists(): return None
        if bands_to_load is None:
            bands_to_load = ["B05","B08","B11","SCL"]
        elif bands_to_load == ["ALL"]:
            bands_to_load = list(band_idxs.keys())
        result = {}
        with rasterio.open(s2_path) as src:
            for b in bands_to_load:
                if b not in band_idxs: continue
                idx = band_idxs[b]
                result[b] = src.read(idx+1) if b == "SCL" else src.read(idx+1).astype(np.float32)
        return result

    def _build_clean_mask(scl, b05, b08, b11, vv_raw, vh_raw, exclude_classes=None):
        if exclude_classes is None: exclude_classes = CLOUD_CLASSES
        s1_valid = np.isfinite(vv_raw) & np.isfinite(vh_raw) & (vv_raw > 0) & (vh_raw > 0)
        clean = (s1_valid & (scl != NODATA_VAL) & (~np.isin(scl, exclude_classes)) &
                 (b05 != NODATA_VAL) & (b08 != NODATA_VAL) & (b11 != NODATA_VAL) &
                 (b05 >= 0) & (b08 >= 0) & (b11 >= 0) &
                 (b05 <= UPPER_BOUND) & (b08 <= UPPER_BOUND) & (b11 <= UPPER_BOUND))
        cloud = np.isin(scl, CLOUD_CLASSES)
        return clean, cloud, s1_valid

    def _clear_sensible(data_dict):
        return (~np.isin(data_dict["SCL"], CLOUD_CLASSES) & (data_dict["SCL"] != NODATA_VAL) &
                _is_sensible(data_dict["B05"]) & _is_sensible(data_dict["B08"]) & _is_sensible(data_dict["B11"]))

    def _preprocess_s1(vv_raw, vh_raw):
        return (10.0 * np.log10(np.clip(vv_raw, VV_CLIP_MIN, VV_CLIP_MAX)),
                10.0 * np.log10(np.clip(vh_raw, VH_CLIP_MIN, VH_CLIP_MAX)))

    GLOBAL_S1_TEXTURE_CACHE_CF = {}

    def _extract_s1_features(vv_db, vh_db, rows, cols):
        global GLOBAL_S1_TEXTURE_CACHE_CF
        key = id(vv_db)
        if key not in GLOBAL_S1_TEXTURE_CACHE_CF:
            GLOBAL_S1_TEXTURE_CACHE_CF.clear()
            cf_log("    [Cache Miss] Computing S1 textures...")
            vv_m3 = ndimage.uniform_filter(vv_db, 3); vh_m3 = ndimage.uniform_filter(vh_db, 3)
            vv_m7 = ndimage.uniform_filter(vv_db, 7); vh_m7 = ndimage.uniform_filter(vh_db, 7)
            vv_var = np.sqrt(np.maximum(ndimage.uniform_filter(vv_db**2,5)-ndimage.uniform_filter(vv_db,5)**2, 0))
            vh_var = np.sqrt(np.maximum(ndimage.uniform_filter(vh_db**2,5)-ndimage.uniform_filter(vh_db,5)**2, 0))
            GLOBAL_S1_TEXTURE_CACHE_CF[key] = {"vv_m3":vv_m3,"vh_m3":vh_m3,"vv_m7":vv_m7,"vh_m7":vh_m7,"vv_var":vv_var,"vh_var":vh_var}
        cache = GLOBAL_S1_TEXTURE_CACHE_CF[key]
        vv = vv_db[rows,cols]; vh = vh_db[rows,cols]
        vv_s = np.maximum(np.abs(vv),1e-10); vh_s = np.maximum(np.abs(vh),1e-10)
        X = np.zeros((len(rows), N_SAR_FEATURES), dtype=np.float32)
        X[:,0]=vv; X[:,1]=vh; X[:,2]=vv-vh; X[:,3]=vh-vv; X[:,4]=vv+vh; X[:,5]=vv; X[:,6]=vh
        X[:,7]=cache["vv_m3"][rows,cols]; X[:,8]=cache["vh_m3"][rows,cols]
        X[:,9]=cache["vv_m7"][rows,cols]; X[:,10]=cache["vh_m7"][rows,cols]
        X[:,11]=cache["vv_var"][rows,cols]; X[:,12]=cache["vh_var"][rows,cols]
        X[:,13]=(vv-vh)/(vv_s+vh_s)
        return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    def _clear_s1_cache_cf():
        global GLOBAL_S1_TEXTURE_CACHE_CF
        GLOBAL_S1_TEXTURE_CACHE_CF.clear()

    def _temporal_fill(proj_date_str, support_dates, band_idxs, rows, cols, target_bands=None):
        fmt = "%Y-%m-%d"; proj_dt = datetime.strptime(proj_date_str, fmt)
        n = len(rows)
        if target_bands is None: target_bands = ["B05","B08","B11"]
        filled = {b: np.full(n, np.nan, dtype=np.float32) for b in target_bands}
        n_obs = np.zeros(n, dtype=np.int8)
        sw=np.zeros(n,dtype=np.float64); swx=np.zeros(n,dtype=np.float64); swxx=np.zeros(n,dtype=np.float64)
        swy={b:np.zeros(n,dtype=np.float64) for b in target_bands}
        swxy={b:np.zeros(n,dtype=np.float64) for b in target_bands}
        for sup_date in support_dates:
            if np.all(n_obs >= HIGH_CONF_MAX_SUPPORT): break
            sup_dt = datetime.strptime(sup_date, fmt)
            days_gap = (proj_dt - sup_dt).days
            if days_gap <= 0: continue
            needed = list(set(target_bands + ["B05","B08","B11","SCL"]))
            sup_data = _load_s2_bands(sup_date, band_idxs, bands_to_load=needed)
            if sup_data is None: continue
            sup_upper = {k.upper(): v for k, v in sup_data.items()}
            if not all(b in sup_upper for b in ["B05","B08","B11","SCL"]):
                del sup_data, sup_upper; gc.collect(); continue
            active = _clear_sensible(sup_upper)[rows, cols] & (n_obs < HIGH_CONF_MAX_SUPPORT)
            if not np.any(active): del sup_data, sup_upper; gc.collect(); continue
            w = 1.0/float(days_gap); x = float(days_gap)
            sw[active]+=w; swx[active]+=w*x; swxx[active]+=w*x*x
            for b in target_bands:
                if b not in sup_upper: continue
                bv = sup_upper[b][rows,cols].astype(np.float64)
                swy[b][active]+=w*bv[active]; swxy[b][active]+=w*x*bv[active]
            n_obs[active]+=1
            del sup_data, sup_upper; gc.collect()
        den = sw*swxx - swx*swx
        valid_reg = (n_obs>=HIGH_CONF_MIN_SUPPORT) & (np.abs(den)>1e-10)
        if np.any(valid_reg):
            for b in target_bands:
                slope = np.clip((sw[valid_reg]*swxy[b][valid_reg] - swx[valid_reg]*swy[b][valid_reg])/den[valid_reg], -500.0, 500.0)
                intercept = (swy[b][valid_reg] - slope*swx[valid_reg]) / np.maximum(sw[valid_reg],1e-10)
                filled[b][valid_reg] = np.clip(np.maximum(intercept,0.0),0.0,UPPER_BOUND).astype(np.float32)
        valid_mean = ((n_obs>=LOW_CONF_MIN_SUPPORT) & (n_obs<=LOW_CONF_MAX_SUPPORT)) | ((n_obs>=HIGH_CONF_MIN_SUPPORT) & (np.abs(den)<=1e-10))
        if np.any(valid_mean):
            for b in target_bands:
                filled[b][valid_mean] = np.clip(swy[b][valid_mean]/np.maximum(sw[valid_mean],1e-10),0.0,UPPER_BOUND).astype(np.float32)
        return filled, n_obs

    def _extract_anchor(target_date_str, candidates, support_dates, band_idxs, rows, cols):
        tgt_dt = datetime.strptime(target_date_str, "%Y-%m-%d"); n = len(rows)
        X_anchor = np.full((n, N_ANCHOR_FEATURES), ANCHOR_SENTINEL, dtype=np.float32)
        cat_x1=np.zeros(n,bool); cat_x2=np.zeros(n,bool); cat_th=np.zeros(n,bool); cat_s=np.zeros(n,bool)
        needs = np.ones(n, bool)
        for ci, cand_date in enumerate(candidates):
            if not np.any(needs): break
            cd = _load_s2_bands(cand_date, band_idxs, bands_to_load=["B05","B08","B11","SCL"])
            if cd is None: continue
            cu = {k.upper(): v for k, v in cd.items()}
            active = needs & _clear_sensible(cu)[rows, cols]
            if np.any(active):
                dg = (tgt_dt - datetime.strptime(cand_date, "%Y-%m-%d")).days
                X_anchor[active,0]=cu["B05"][rows[active],cols[active]]
                X_anchor[active,1]=cu["B08"][rows[active],cols[active]]
                X_anchor[active,2]=cu["B11"][rows[active],cols[active]]
                X_anchor[active,3]=float(dg)
                (cat_x1 if ci==0 else cat_x2)[active] = True
                needs[active] = False
            del cd, cu; gc.collect()
        if np.any(needs) and len(support_dates) > 0:
            pg = (tgt_dt - datetime.strptime(candidates[0], "%Y-%m-%d")).days
            fb, sc = _temporal_fill(candidates[0], support_dates, band_idxs, rows[needs], cols[needs], ["B05","B08","B11"])
            sh = sc >= HIGH_CONF_MIN_SUPPORT
            ti = np.where(needs)[0]
            if np.any(sh):
                sg = ti[sh]
                X_anchor[sg,0]=fb["B05"][sh]; X_anchor[sg,1]=fb["B08"][sh]; X_anchor[sg,2]=fb["B11"][sh]; X_anchor[sg,3]=float(pg)
                cat_th[sg]=True; needs[sg]=False
        cat_s[needs] = True
        return X_anchor, {"direct_x1_count":int(cat_x1.sum()),"direct_x2_count":int(cat_x2.sum()),"temporal_high_count":int(cat_th.sum()),"sentinel_count":int(cat_s.sum()),"total":n}

    def _print_tracker(tracker, label=""):
        total = tracker["total"]
        cf_log(f"  {'─'*50}")
        if label: cf_log(f"  Anchor Resolution — {label}")
        cf_log(f"  Total pixels              : {total:>12,}")
        cf_log(f"  Direct X-1                : {tracker['direct_x1_count']:>12,} ({tracker['direct_x1_count']/total*100:.2f}%)")
        cf_log(f"  Direct X-2                : {tracker['direct_x2_count']:>12,} ({tracker['direct_x2_count']/total*100:.2f}%)")
        cf_log(f"  Temporal High (4-6 obs)   : {tracker['temporal_high_count']:>12,} ({tracker['temporal_high_count']/total*100:.2f}%)")
        cf_log(f"  Sentinel -1 (pure SAR)    : {tracker['sentinel_count']:>12,} ({tracker['sentinel_count']/total*100:.2f}%)")
        cf_log(f"  {'─'*50}")

    # ── CELL 3A: Auto date detection ───────────────────────────────────────
    cf_log(""); cf_log("="*70); cf_log("AUTO DATE DETECTION"); cf_log("="*70)
    if not CF_PAIRS_DIR.exists():
        raise RuntimeError(f"PAIRS_DIR not found: {CF_PAIRS_DIR}")
    dp = re.compile(r"(\d{4}-\d{2}-\d{2})")
    fd = []
    for folder in CF_PAIRS_DIR.iterdir():
        if not folder.is_dir(): continue
        mm = dp.search(folder.name)
        if mm:
            try: datetime.strptime(mm.group(1),"%Y-%m-%d"); fd.append(mm.group(1))
            except ValueError: continue
    if not fd:
        raise RuntimeError(f"No date folders in {CF_PAIRS_DIR}")
    fd = sorted(set(fd), reverse=True)
    cf_log(f"  Date folders found  : {len(fd)}")
    cf_log(f"  Newest              : {fd[0]}")
    cf_log(f"  Oldest              : {fd[-1]}")
    if TARGET_DATE_CF not in fd:
        raise RuntimeError(f"TARGET_DATE {TARGET_DATE_CF} not in PAIRS_DIR. Available: {fd}")
    ti = fd.index(TARGET_DATE_CF)
    ALL_DATES_CF = fd[ti:min(ti+13, len(fd))]
    TRAIN_DATES_CF = ALL_DATES_CF[1:N_TRAIN_DATES_CF+1]
    cf_log(f"  Working pool        : {ALL_DATES_CF}")
    cf_log(f"  Train dates         : {TRAIN_DATES_CF}")
    ANCHOR_CANDS_CF = {}; TIER1_SUP_CF = {}; TIER2_SUP_CF = {}
    for i, cd in enumerate(ALL_DATES_CF):
        TIER2_SUP_CF[cd] = ALL_DATES_CF[i+1:min(i+1+MAX_LOOKBACK_WINDOW, len(ALL_DATES_CF))]
        cands = [ALL_DATES_CF[j] for j in [i+1,i+2] if j < len(ALL_DATES_CF)]
        ANCHOR_CANDS_CF[cd] = cands
        TIER1_SUP_CF[cd] = ALL_DATES_CF[i+3:min(i+3+MAX_LOOKBACK_WINDOW, len(ALL_DATES_CF))]
    cf_log(f"  Inference anchors   : {ANCHOR_CANDS_CF.get(TARGET_DATE_CF,[])}")
    cf_log(f"  Tier1 support       : {TIER1_SUP_CF.get(TARGET_DATE_CF,[])}")

    # Band detection
    s2_tgt_path_cf, _ = _get_s2s1_paths(TARGET_DATE_CF)
    if not s2_tgt_path_cf:
        raise RuntimeError(f"S2 not found for {TARGET_DATE_CF}")
    band_idxs_cf = _detect_bands(s2_tgt_path_cf)
    cf_log(f"  Band indices        : {band_idxs_cf}")

    # ── CELL 3B: Training ──────────────────────────────────────────────────
    cf_log(""); cf_log("="*70); cf_log("TRAINING RANDOM FOREST MODELS"); cf_log("="*70)
    t_train = time.time()
    np.random.seed(RANDOM_STATE_CF)
    budget = MAX_SAMPLES_CF // max(len(TRAIN_DATES_CF), 1)
    X_tr_list = []; y_tr_dict = {b:[] for b in REFLECTIVE_BANDS}
    g_tracker = {"direct_x1_count":0,"direct_x2_count":0,"temporal_high_count":0,"sentinel_count":0,"total":0}

    for ds in TRAIN_DATES_CF:
        cf_log(f"\n  Training date: {ds}")
        s2p, s1p = _get_s2s1_paths(ds)
        if not s2p or not s1p:
            cf_log(f"    WARNING: S2 or S1 missing for {ds}. Skipping."); continue
        with rasterio.open(s1p) as src_s1:
            vv_r = src_s1.read(1).astype(np.float32)
            vh_r = src_s1.read(2).astype(np.float32)
        s2d = _load_s2_bands(ds, band_idxs_cf, REFLECTIVE_BANDS+["SCL"])
        if s2d is None:
            del vv_r, vh_r; gc.collect(); continue
        b05=s2d.get("B05",np.zeros_like(vv_r)); b08=s2d.get("B08",np.zeros_like(vv_r))
        b11=s2d.get("B11",np.zeros_like(vv_r)); scl=s2d.get("SCL",np.zeros_like(vv_r,dtype=np.int32))
        cm, _, _ = _build_clean_mask(scl, b05, b08, b11, vv_r, vh_r, EFFECTIVE_TRAIN_EXCLUDE)
        cr, cc = np.where(cm); n_clean = len(cr)
        if n_clean == 0:
            del vv_r,vh_r,s2d,b05,b08,b11,scl; gc.collect(); continue
        n_sample = min(int(n_clean*SAMPLE_FRACTION), budget)
        idx = np.random.choice(n_clean, n_sample, replace=False)
        sr, sc_idx = cr[idx], cc[idx]
        cf_log(f"    Clean: {n_clean:,} | Sampled: {n_sample:,}")
        vv_db_tr, vh_db_tr = _preprocess_s1(vv_r, vh_r)
        X_sar_tr = _extract_s1_features(vv_db_tr, vh_db_tr, sr, sc_idx)
        cands_tr = ANCHOR_CANDS_CF.get(ds, []); sup_tr = TIER1_SUP_CF.get(ds, [])
        X_anc_tr, trk = _extract_anchor(ds, cands_tr, sup_tr, band_idxs_cf, sr, sc_idx)
        _print_tracker(trk, ds)
        for key in g_tracker: g_tracker[key] += trk.get(key, 0)
        X_tr_list.append(np.hstack([X_sar_tr, X_anc_tr]))
        for b in REFLECTIVE_BANDS:
            y_tr_dict[b].append(s2d[b][sr,sc_idx].astype(np.float32) if b in s2d else np.zeros(n_sample,np.float32))
        del vv_r,vh_r,vv_db_tr,vh_db_tr,b05,b08,b11,scl,s2d,X_sar_tr,X_anc_tr; gc.collect()

    if not X_tr_list:
        raise RuntimeError("No training data collected. Check S1/S2 files in pairs/.")
    X_train_cf = np.vstack(X_tr_list); del X_tr_list; gc.collect()
    cf_log(""); cf_log("="*70); cf_log("GLOBAL TRAINING ANCHOR SUMMARY"); cf_log("="*70)
    _print_tracker(g_tracker, "All Training Dates")
    cf_log(f"  Total samples : {X_train_cf.shape[0]:,}  |  Features : {X_train_cf.shape[1]}")
    cf_log(""); cf_log("="*70); cf_log("FITTING RANDOM FOREST MODELS"); cf_log("="*70)
    models_cf = {}
    for b in REFLECTIVE_BANDS:
        cf_log(f"  Band {b} ...")
        t_b = time.time()
        y_tr = np.nan_to_num(np.concatenate(y_tr_dict[b]), nan=0.0)
        rf = RandomForestRegressor(n_estimators=RF_N_ESTIMATORS_CF, max_depth=RF_MAX_DEPTH_CF,
                                   min_samples_leaf=RF_MIN_SAMPLES_LEAF_CF, max_features=RF_MAX_FEATURES_CF,
                                   n_jobs=RF_N_JOBS_CF, random_state=RANDOM_STATE_CF)
        rf.fit(X_train_cf, y_tr); models_cf[b] = rf
        cf_log(f"    Trained in {time.time()-t_b:.1f}s | {len(y_tr):,} samples")
        del y_tr; gc.collect()
    _clear_s1_cache_cf(); del X_train_cf; gc.collect()
    cf_log(f"\n  All {len(models_cf)} RF models trained in {time.time()-t_train:.1f}s")

    # ── CELL 3C: Inference ─────────────────────────────────────────────────
    cf_log(""); cf_log("="*70); cf_log(f"5-TIER INFERENCE — {TARGET_DATE_CF}"); cf_log("="*70)
    t_inf = time.time()
    s2_tgt_path_cf, s1_tgt_path_cf = _get_s2s1_paths(TARGET_DATE_CF)
    if not s2_tgt_path_cf: raise RuntimeError(f"S2 not found for {TARGET_DATE_CF}")
    if not s1_tgt_path_cf: raise RuntimeError(f"S1 not found for {TARGET_DATE_CF}")
    cf_log(f"  S2: {s2_tgt_path_cf.name}")
    cf_log(f"  S1: {s1_tgt_path_cf.name}")
    with rasterio.open(s1_tgt_path_cf) as src_s1:
        vv_raw_cf = src_s1.read(1).astype(np.float32)
        vh_raw_cf = src_s1.read(2).astype(np.float32)
        h_cf, w_cf = vv_raw_cf.shape
    tgt_bands_cf = _load_s2_bands(TARGET_DATE_CF, band_idxs_cf, REFLECTIVE_BANDS+AUXILIARY_BANDS)
    if tgt_bands_cf is None: raise RuntimeError(f"Cannot load S2 bands for {TARGET_DATE_CF}")
    with rasterio.open(s2_tgt_path_cf) as src_s2_tgt:
        base_profile_cf = src_s2_tgt.profile.copy()
    b05_cf=tgt_bands_cf.get("B05",np.zeros((h_cf,w_cf),np.float32))
    b08_cf=tgt_bands_cf.get("B08",np.zeros((h_cf,w_cf),np.float32))
    b11_cf=tgt_bands_cf.get("B11",np.zeros((h_cf,w_cf),np.float32))
    scl_cf=tgt_bands_cf.get("SCL",np.zeros((h_cf,w_cf),np.int32))
    tgt_clean_mask_cf, tgt_cloud_mask_cf, s1_valid_tgt_cf = _build_clean_mask(scl_cf,b05_cf,b08_cf,b11_cf,vv_raw_cf,vh_raw_cf,CLOUD_CLASSES)
    tgt_nodata_mask_cf = ((scl_cf==NODATA_VAL)|(b05_cf==NODATA_VAL)|(b08_cf==NODATA_VAL)|(b11_cf==NODATA_VAL)|(b05_cf<0)|(b08_cf<0)|(b11_cf<0))
    cf_log(f"  Cloud pixels : {np.sum(tgt_cloud_mask_cf):,} ({np.sum(tgt_cloud_mask_cf)/(h_cf*w_cf)*100:.2f}%)")
    cf_log(f"  Valid S1     : {np.sum(s1_valid_tgt_cf):,} ({np.sum(s1_valid_tgt_cf)/(h_cf*w_cf)*100:.2f}%)")
    vv_db_cf, vh_db_cf = _preprocess_s1(vv_raw_cf, vh_raw_cf)
    del vv_raw_cf, vh_raw_cf; gc.collect()
    pred_bands_cf = {b: np.full((h_cf,w_cf), np.nan, np.float32) for b in REFLECTIVE_BANDS}
    tier_map_cf = np.zeros((h_cf,w_cf), np.uint8)

    # Phase 1A: Anchor extraction
    pr_cf, pc_cf = np.where(s1_valid_tgt_cf)
    cf_log(f"\nPHASE 1A: Anchor extraction ({len(pr_cf):,} S1-valid pixels)")
    cands_inf  = ANCHOR_CANDS_CF.get(TARGET_DATE_CF,[])
    sup_inf    = TIER1_SUP_CF.get(TARGET_DATE_CF,[])
    X_anc_full_cf, inf_trk = _extract_anchor(TARGET_DATE_CF, cands_inf, sup_inf, band_idxs_cf, pr_cf, pc_cf)
    _print_tracker(inf_trk, f"Inference Anchors — {TARGET_DATE_CF}")

    # Phase 1B: RF prediction in chunks
    cf_log(f"\nPHASE 1B: RF prediction (chunk size {CHUNK_SIZE_CF:,})")
    for cs in tqdm(range(0, len(pr_cf), CHUNK_SIZE_CF), desc="  RF Prediction"):
        ce = min(cs+CHUNK_SIZE_CF, len(pr_cf))
        r_ch=pr_cf[cs:ce]; c_ch=pc_cf[cs:ce]
        X_sar_ch = _extract_s1_features(vv_db_cf, vh_db_cf, r_ch, c_ch)
        X_anc_ch = X_anc_full_cf[cs:ce]
        X_chunk  = np.hstack([X_sar_ch, X_anc_ch])
        for b in REFLECTIVE_BANDS:
            if b not in models_cf: continue
            pred_bands_cf[b][r_ch,c_ch] = np.clip(models_cf[b].predict(X_chunk),0,UPPER_BOUND).astype(np.float32)
        valid_anc = X_anc_ch[:,0] != ANCHOR_SENTINEL
        tier_map_cf[r_ch[valid_anc], c_ch[valid_anc]] = TIER_1_FUSION_HIGH
        tier_map_cf[r_ch[~valid_anc],c_ch[~valid_anc]] = TIER_3_FUSION_LOW
        del X_sar_ch, X_anc_ch, X_chunk; gc.collect()
    del X_anc_full_cf, vv_db_cf, vh_db_cf; gc.collect()

    # Phase 2: Temporal fill
    needs_t = np.isnan(pred_bands_cf[REFLECTIVE_BANDS[0]]); rt_cf, ct_cf = np.where(needs_t)
    if len(rt_cf) > 0:
        cf_log(f"\nPHASE 2: Temporal fill ({len(rt_cf):,} pixels)")
        t2_sup = TIER2_SUP_CF.get(TARGET_DATE_CF,[])
        fb_t, t_obs = _temporal_fill(TARGET_DATE_CF, t2_sup, band_idxs_cf, rt_cf, ct_cf, REFLECTIVE_BANDS)
        success = t_obs >= LOW_CONF_MIN_SUPPORT
        if np.any(success):
            for b in REFLECTIVE_BANDS:
                pred_bands_cf[b][rt_cf[success],ct_cf[success]] = fb_t[b][success]
            hc = (t_obs[success] >= HIGH_CONF_MIN_SUPPORT)
            tier_map_cf[rt_cf[success][hc], ct_cf[success][hc]] = TIER_2_TEMPORAL_HIGH
            tier_map_cf[rt_cf[success][~hc],ct_cf[success][~hc]] = TIER_4_TEMPORAL_LOW
        del fb_t, t_obs; gc.collect()
    else:
        cf_log("\nPHASE 2: Skipped — all pixels had valid S1.")

    # Phase 3: Spatial interpolation
    needs_t5 = np.isnan(pred_bands_cf[REFLECTIVE_BANDS[0]])
    if np.any(needs_t5):
        cf_log(f"\nPHASE 3: Spatial interpolation ({np.sum(needs_t5):,} remaining gaps)")
        _, idx_edt = ndimage.distance_transform_edt(needs_t5, return_distances=True, return_indices=True)
        for b in REFLECTIVE_BANDS:
            filled = pred_bands_cf[b].copy()
            filled[needs_t5] = pred_bands_cf[b][idx_edt[0][needs_t5], idx_edt[1][needs_t5]]
            pred_bands_cf[b] = filled
        tier_map_cf[needs_t5] = TIER_5_SPATIAL
    else:
        cf_log("\nPHASE 3: Skipped.")
    _clear_s1_cache_cf()

    total_px = h_cf*w_cf
    cf_log("\nPIXEL TIER DISTRIBUTION:")
    for tid, label in [(1,"Fusion High"),(2,"Temporal High"),(3,"Fusion Low"),(4,"Temporal Low"),(5,"Spatial")]:
        c = int(np.sum(tier_map_cf == tid))
        cf_log(f"  Tier {tid} ({label:<15}) : {c:>10,} ({c/total_px*100:.2f}%)")
    cf_log(f"\n  Total inference time : {time.time()-t_inf:.1f}s")

    # ── CELL 3D: Spectral Indices ──────────────────────────────────────────
    def _compute_idx(b1, b2, eps=1e-6):
        return np.clip((b2-b1)/(b2+b1+eps),-1.0,1.0).astype(np.float32)

    pred_b05_cf = pred_bands_cf.get("B05"); pred_b08_cf = pred_bands_cf.get("B08"); pred_b11_cf = pred_bands_cf.get("B11")
    pred_ndre_cf = _compute_idx(pred_b05_cf, pred_b08_cf)
    pred_ndwi_cf = _compute_idx(pred_b11_cf, pred_b08_cf)
    tgt_b05_cf = tgt_bands_cf.get("B05"); tgt_b08_cf = tgt_bands_cf.get("B08"); tgt_b11_cf = tgt_bands_cf.get("B11")
    tgt_ndre_cf = _compute_idx(tgt_b05_cf, tgt_b08_cf); tgt_ndwi_cf = _compute_idx(tgt_b11_cf, tgt_b08_cf)
    clear_inf = (~tgt_cloud_mask_cf) & (~tgt_nodata_mask_cf)
    fused_ndre_cf = np.where(clear_inf, tgt_ndre_cf, pred_ndre_cf).astype(np.float32)
    fused_ndwi_cf = np.where(clear_inf, tgt_ndwi_cf, pred_ndwi_cf).astype(np.float32)
    cf_log(f"\n  Fused NDRE: mean={float(np.nanmean(fused_ndre_cf)):.4f}  Fused NDWI: mean={float(np.nanmean(fused_ndwi_cf)):.4f}")

    # Fix nodata clamping (from rs-curve-update.ipynb)
    NODATA_IDX = -32768.0
    def _clamp_index(data, name):
        out = data.copy()
        nd_mask = np.isnan(out); out[nd_mask] = NODATA_IDX
        valid = out != NODATA_IDX
        if name == "NDWI":
            out[valid & (out < -0.9)] = -0.9
            out[valid & (out >  0.349)] = 0.31
        elif name == "NDRE":
            out[valid & (out < 0)] = 0.1
            out[valid & (out > 0.7)] = 0.69
        return out.astype(np.float32)
    fused_ndre_cf = _clamp_index(fused_ndre_cf, "NDRE")
    fused_ndwi_cf = _clamp_index(fused_ndwi_cf, "NDWI")
    cf_log("  Value clamping applied to NDRE and NDWI.")

    # ── CELL 3E: Export TIFFs ──────────────────────────────────────────────
    cf_log(""); cf_log("="*70); cf_log("EXPORTING TIFFs"); cf_log("="*70)
    t_ex = time.time()

    def _make_filename(product, suffix=None):
        dc = TARGET_DATE_CF.replace("-","_")
        if product == "S2_Fused": return f"cloud_free_{TARGET_DATE_CF}.tif"
        if suffix == "predicted":  return f"{product.lower()}_PROD_{dc}_predicted.tif"
        return f"{product.lower()}_PROD_{dc}.tif"

    def _save_tif1(data, filename, dtype="float32", nodata=OUTPUT_NODATA, desc=""):
        fp = TIFF_DIR_CF / filename
        prof = base_profile_cf.copy(); prof.update(dtype=dtype, count=1, nodata=nodata, compress="lzw")
        out = data.copy()
        if dtype == "float32": out[~np.isfinite(out)] = nodata
        else: out = np.nan_to_num(out, nan=nodata).astype(np.int32)
        with rasterio.open(fp,"w",**prof) as dst:
            dst.write(out.astype(dtype),1)
            if desc: dst.update_tags(1, name=desc)
        cf_log(f"  ✅ {filename} ({fp.stat().st_size/1024/1024:.1f} MB)")
        return fp

    def _save_tif_multi(band_arrs, band_names, filename, dtype="int32", nodata=OUTPUT_NODATA):
        fp = TIFF_DIR_CF / filename
        prof = base_profile_cf.copy(); prof.update(dtype=dtype, count=len(band_names), nodata=nodata, compress="lzw")
        with rasterio.open(fp,"w",**prof) as dst:
            for bi, bn in enumerate(band_names, 1):
                arr = band_arrs.get(bn, np.full((h_cf,w_cf),nodata,dtype=dtype))
                out = arr.copy()
                if dtype == "float32": out[~np.isfinite(out)] = nodata
                else: out = np.nan_to_num(out, nan=nodata).astype(np.int32)
                dst.write(out.astype(dtype), bi); dst.update_tags(bi, name=bn)
        cf_log(f"  ✅ {filename} ({fp.stat().st_size/1024/1024:.1f} MB)")
        return fp

    # 1. NDRE cloudfill (fused)
    ndre_fused_path = _save_tif1(fused_ndre_cf, _make_filename("NDRE","cloudfill"), desc="NDRE_fused")
    # 2. NDWI cloudfill (fused)
    ndwi_fused_path = _save_tif1(fused_ndwi_cf, _make_filename("NDWI","cloudfill"), desc="NDWI_fused")
    # 3. NDRE predicted
    _save_tif1(pred_ndre_cf, _make_filename("NDRE","predicted"), desc="NDRE_predicted")
    # 4. NDWI predicted
    _save_tif1(pred_ndwi_cf, _make_filename("NDWI","predicted"), desc="NDWI_predicted")
    # 5. Full S2 fused (15 bands)
    fused_s2 = {}
    for b in REFLECTIVE_BANDS:
        ta=tgt_bands_cf.get(b); pa=pred_bands_cf.get(b)
        if ta is not None and pa is not None:
            fused_s2[b] = np.where(clear_inf, ta, pa).astype(np.float32)
    for b in AUXILIARY_BANDS:
        if b in tgt_bands_cf: fused_s2[b] = tgt_bands_cf[b]
    _save_tif_multi(fused_s2, ALL_BANDS_ORDERED, _make_filename("S2_Fused","cloudfill"))
    del fused_s2; gc.collect()
    # 6. Tier map
    tier_profile = base_profile_cf.copy(); tier_profile.update(dtype="uint8",count=1,nodata=255,compress="lzw")
    tier_fp = TIFF_DIR_CF / f"tier_map_{TARGET_DATE_CF}.tif"
    with rasterio.open(tier_fp,"w",**tier_profile) as dst: dst.write(tier_map_cf, 1)
    cf_log(f"  ✅ tier_map_{TARGET_DATE_CF}.tif ({tier_fp.stat().st_size/1024/1024:.1f} MB)")
    # 7. Synthetic (optional)
    if SAVE_SYNTHETIC_FULL:
        synth = {b: pred_bands_cf[b] for b in REFLECTIVE_BANDS}
        for b in AUXILIARY_BANDS:
            if b in tgt_bands_cf: synth[b] = tgt_bands_cf[b]
        _save_tif_multi(synth, ALL_BANDS_ORDERED, _make_filename("S2_Synthetic","predicted"))

    cf_log(f"  Export time : {time.time()-t_ex:.1f}s")
    cf_log(f"  TIFFs saved to : {TIFF_DIR_CF}")

    # ── Validation (optional) ──────────────────────────────────────────────
    if RUN_VALIDATION:
        cf_log("\n⚠️  RUN_VALIDATION = True — validation metrics computation goes here.")
        cf_log("   (See All_Mills.ipynb Cell 10 for the full validation block.)")

    # ── Export key variables ───────────────────────────────────────────────
    CLOUDFILL_TIFF_DIR  = TIFF_DIR_CF
    NDRE_CLOUDFILL_PATH = ndre_fused_path
    NDWI_CLOUDFILL_PATH = ndwi_fused_path

    cf_log(""); cf_log("="*70); cf_log("CLOUD-FILL PIPELINE COMPLETE"); cf_log("="*70)
    cf_log(f"  NDRE cloudfill : {NDRE_CLOUDFILL_PATH}")
    cf_log(f"  NDWI cloudfill : {NDWI_CLOUDFILL_PATH}")
    cf_log(f"  Log file       : {_log_path}")
    print(f"\n  ✅ CLOUDFILL COMPLETE | NDRE: {NDRE_CLOUDFILL_PATH.name} | NDWI: {NDWI_CLOUDFILL_PATH.name}")


## 🌱 Cell 4 — KPI Processing

Clips the cloud-free NDRE and NDWI rasters to each parcel in `PARCELS_GEOJSON` and computes:

| KPI | Output file | Key columns |
|-----|------------|-------------|
| **Health** | `Jalna_DATA_HEALTH.parquet` | `ndre_mean`, `ndwi_mean`, `chi_score` (0-100), 5 NDRE area buckets (ha) |
| **Water Stress** | `Jalna_DATA_WATER.parquet` | `ndwi_mean`, 5 NDWI area buckets (ha) |
| **Harvest / Maturity** | `Jalna_DATA_HARVEST.parquet` | `days_since_planting`, `mri_score` (0-100) |
| **Fertilizer (Nitrogen)** | `Jalna_DATA_FERTILIZER.parquet` | `ndi_score`, `n_rate_kg_ha`, `vra_priority`, `stress_type` |
| **Weed** | `Jalna_DATA_WEED.parquet` | `weed_area_ha`, `weed_pct`, `weed_pressure_index`, `alert_level` |

Extracted per-parcel GeoTIFFs are also saved to `KPI_OUTPUT_DIR`.


In [ ]:
# ============================================================================
# CELL 4: KPI PROCESSING PIPELINE
# ============================================================================
# Source: Processing_Code_new.ipynb (main pipeline + weed module)
#
# Reads NDRE/NDWI TIFFs from CLOUDFILL_TIFF_DIR and parcel GeoJSON
# from PARCELS_GEOJSON (set in Cell 0).
# Writes parquet KPI files and extracted TIFFs to KPI_OUTPUT_DIR.
# ============================================================================

if not RUN_KPI:
    print("⏭️  RUN_KPI = False — skipping KPI processing stage.")
else:
    print("="*70); print("KPI PROCESSING PIPELINE"); print("="*70)

    # ── Resolve input raster paths ────────────────────────────────────────
    # Build the expected filenames based on INFERENCE_DATE (set in Cell 2)
    _dc = INFERENCE_DATE.replace("-","_")  # e.g. "2026_05_03"

    # Look for NDRE/NDWI cloudfill TIFFs in the cloud-fill output dir
    def _find_raster(tiff_dir, pattern):
        """Find the first file matching pattern in tiff_dir, or None."""
        tiff_dir = Path(tiff_dir)
        if not tiff_dir.exists():
            return None
        candidates = list(tiff_dir.glob(pattern))
        return candidates[0] if candidates else None

    _ndre_input = _find_raster(CLOUDFILL_TIFF_DIR, f"*ndre*{_dc}*.tif") or                   _find_raster(CLOUDFILL_TIFF_DIR, "*NDRE*cloudfill*.tif")
    _ndwi_input = _find_raster(CLOUDFILL_TIFF_DIR, f"*ndwi*{_dc}*.tif") or                   _find_raster(CLOUDFILL_TIFF_DIR, "*NDWI*cloudfill*.tif")

    if not _ndre_input or not _ndwi_input:
        raise FileNotFoundError(
            f"Cannot find NDRE/NDWI cloudfill TIFFs in {CLOUDFILL_TIFF_DIR}.\n"
            f"  NDRE found : {_ndre_input}\n"
            f"  NDWI found : {_ndwi_input}\n"
            "  Check that Cell 3 ran successfully, or set RUN_CLOUDFILL=True."
        )

    print(f"  NDRE input : {_ndre_input.name}")
    print(f"  NDWI input : {_ndwi_input.name}")

    # ── Sowing dates loader ───────────────────────────────────────────────
    def _load_sowing_dates(parcels_gdf):
        """
        Load per-farmer sowing info from JSON.
        If the file does not exist, create a default one using
        DEFAULT_PLANTING_DATE and DEFAULT_CYCLE_DAYS from Cell 0.
        """
        if SOWING_DATES_PATH.exists():
            with open(SOWING_DATES_PATH) as f:
                data = json.load(f)
        else:
            data = {}
            for sr in parcels_gdf["sr_no"].unique():
                data[str(int(sr))] = {
                    "planting_date": DEFAULT_PLANTING_DATE,
                    "total_cycle_days": DEFAULT_CYCLE_DAYS,
                    "variety": ""
                }
            SOWING_DATES_PATH.parent.mkdir(parents=True, exist_ok=True)
            with open(SOWING_DATES_PATH, "w") as f:
                json.dump(data, f, indent=2)
            print(f"  ℹ️  sowing_dates.json not found — created default at {SOWING_DATES_PATH}")
        return data

    # ── KPI formulae ──────────────────────────────────────────────────────
    def _compute_CHI(ndre_mean, ndwi_mean):
        """
        Simplified Composite Health Index (0-100).
        Gaussian membership to optimal NDRE (0.50) and NDWI (0.12).
        Weights: NDRE 65%, NDWI 35%.
        """
        ndre_score = np.exp(-0.5 * ((ndre_mean - 0.50) / 0.15)**2)
        ndwi_score = np.exp(-0.5 * ((ndwi_mean - 0.12) / 0.12)**2)
        return float(np.clip(round((0.65*ndre_score + 0.35*ndwi_score)*100, 2), 0, 100))

    def _compute_MRI(dap, ndwi_mean, total_cycle=450):
        """
        Maturity Readiness Index (0-100).
        Sigmoid time component + linear NDWI dry-down.
        """
        inflection = 0.60 * total_cycle
        T_score = 100 / (1 + np.exp(-0.015*(dap - inflection)))
        W_score = float(np.clip((0.20 - ndwi_mean) / 0.40 * 100, 0, 100))
        w_wt = float(np.clip(dap/total_cycle, 0, 1)) * 0.50
        return float(np.clip(round((1-w_wt)*T_score + w_wt*W_score, 2), 0, 100))

    def _classify_stress(ndre_mean, ndwi_mean):
        low_ndre = ndre_mean < 0.35
        low_ndwi = ndwi_mean < 0.0
        if low_ndre and low_ndwi: return "WATER_FIRST"
        elif low_ndre:            return "N_DEFICIENT"
        elif low_ndwi:            return "WATER_STRESS_ONLY"
        else:                     return "SUFFICIENT"

    def _compute_NDI_VRA(ndre_mean, ndwi_mean, ndre_std, field_area_ha, dap):
        """Nitrogen Demand Index and Variable Rate Application priority."""
        stress = _classify_stress(ndre_mean, ndwi_mean)
        base_N = 60
        if stress in ("WATER_FIRST","WATER_STRESS_ONLY","SUFFICIENT"):
            ndi = 0.0 if stress=="SUFFICIENT" else 15.0
            return {"stress_type":stress,"NDI":ndi,"VRA_priority":"NO_N","N_rate_kg_ha":0.0}
        shortfall = max(0, 0.50 - ndre_mean)
        SI = shortfall / 0.50
        ndi = float(np.clip(SI*100, 0, 100))
        n_rate = float(np.clip(base_N*SI, 0, base_N*1.2))
        if dap > 330: n_rate = 0.0; ndi = max(0, ndi-30)
        px = field_area_ha * 10000 / 100
        thr = 0.15 if px < 50 else 0.10
        if ndre_mean < 0.35 and ndre_std < thr:  vra = "UNIFORM_HIGH"
        elif ndre_mean < 0.35:                    vra = "VRA_NEEDED"
        elif ndre_mean < 0.45 and ndre_std >= thr: vra = "VRA_NEEDED"
        elif ndre_mean < 0.45:                    vra = "UNIFORM_LOW"
        else:                                     vra = "NO_N"; n_rate=0.0; ndi=0.0
        return {"stress_type":stress,"NDI":round(ndi,1),"VRA_priority":vra,"N_rate_kg_ha":round(n_rate,1)}

    # ── Bucket area helpers ────────────────────────────────────────────────
    _NDRE_BUCKETS = [("very_poor",-999,0.20),("poor",0.20,0.35),("moderate",0.35,0.45),("good",0.45,0.60),("excellent",0.60,999)]
    _NDWI_BUCKETS = [("very_dry",-999,-0.20),("dry",-0.20,0.00),("moderate",0.00,0.10),("wet",0.10,0.20),("very_wet",0.20,999)]

    def _bucket_areas(pixels, px_to_ha, buckets):
        return {name: np.sum((pixels >= lo) & (pixels < hi)) * px_to_ha for name, lo, hi in buckets}

    # ── Main processing per date folder ───────────────────────────────────
    def _process_date_folder(date_str, ndre_path, ndwi_path, parcels_gdf, sowing_data):
        """
        Clips NDRE and NDWI rasters to each parcel and computes all KPIs.
        Returns four DataFrames: health, water, harvest, fertilizer.
        """
        print(f"\n📅 Processing KPIs for: {date_str}")
        current_date = pd.to_datetime(date_str.replace("_","-"))

        # Reproject parcels to raster CRS dynamically
        with rasterio.open(ndre_path) as src:
            raster_crs = src.crs
        parcels_utm = parcels_gdf.to_crs(raster_crs)

        health_rows, water_rows, harvest_rows, fert_rows = [], [], [], []

        for idx, row in parcels_utm.iterrows():
            parcel_id = row["parcel_id"]
            sr_no     = str(row["sr_no"])
            info      = sowing_data.get(sr_no, {"planting_date": DEFAULT_PLANTING_DATE,
                                                 "last_planting_date": DEFAULT_PLANTING_DATE,
                                                 "total_cycle_days": DEFAULT_CYCLE_DAYS})
            planting_dt      = pd.to_datetime(info.get("planting_date", DEFAULT_PLANTING_DATE))
            last_planting_dt = pd.to_datetime(info.get("last_planting_date", info.get("planting_date", DEFAULT_PLANTING_DATE)))
            eff_planting = planting_dt if current_date >= planting_dt else last_planting_dt
            dap          = (current_date - eff_planting).days
            total_cycle  = info.get("total_cycle_days", DEFAULT_CYCLE_DAYS)
            geom = [row.geometry]

            try:
                # ── NDRE extraction ────────────────────────────────────────
                with rasterio.open(ndre_path) as src:
                    out_ndre, out_tr_ndre = rasterio_mask(src, geom, crop=True, all_touched=False)
                    ndre_nd = src.nodata if src.nodata is not None else 0
                    meta_ndre = src.meta.copy()
                valid_ndre = out_ndre[0][out_ndre[0] != ndre_nd]
                if len(valid_ndre) == 0: continue

                # ── NDWI extraction ────────────────────────────────────────
                with rasterio.open(ndwi_path) as src:
                    out_ndwi, out_tr_ndwi = rasterio_mask(src, geom, crop=True, all_touched=False)
                    ndwi_nd = src.nodata if src.nodata is not None else 0
                    meta_ndwi = src.meta.copy()
                valid_ndwi = out_ndwi[0][out_ndwi[0] != ndwi_nd]
                if len(valid_ndwi) == 0: continue

                total_pix  = len(valid_ndre)
                px_to_ha   = row["area_ha"] / total_pix if total_pix > 0 else 0
                ndre_mean  = float(np.mean(valid_ndre)); ndre_std = float(np.std(valid_ndre))
                ndwi_mean  = float(np.mean(valid_ndwi)); ndwi_std = float(np.std(valid_ndwi))

                # ── Updated : Save extracted TIFFs ───────────────────────────────────
                # Create a date-specific subfolder inside KPI_OUTPUT_DIR
                date_subfolder = KPI_OUTPUT_DIR / date_str
                date_subfolder.mkdir(parents=True, exist_ok=True)

                # Save extracted TIFFs inside that subfolder
                meta_ndre.update({"height":out_ndre.shape[1],"width":out_ndre.shape[2],"transform":out_tr_ndre})
                with rasterio.open(date_subfolder/f"{parcel_id}_NDRE_{date_str}_extracted.tif","w",**meta_ndre) as dst: dst.write(out_ndre)
                
                meta_ndwi.update({"height":out_ndwi.shape[1],"width":out_ndwi.shape[2],"transform":out_tr_ndwi})
                with rasterio.open(date_subfolder/f"{parcel_id}_NDWI_{date_str}_extracted.tif","w",**meta_ndwi) as dst: dst.write(out_ndwi)

                # ── KPI 1: Health ──────────────────────────────────────────
                chi = _compute_CHI(ndre_mean, ndwi_mean)
                ndre_areas = _bucket_areas(valid_ndre, px_to_ha, _NDRE_BUCKETS)
                health_rows.append({"date":date_str,"parcel_id":parcel_id,"days_since_planting":dap,
                    "ndre_mean":ndre_mean,"ndre_stdv":ndre_std,"ndwi_mean":ndwi_mean,"chi_score":chi,
                    "area_verypoor_ha":ndre_areas["very_poor"],"area_poor_ha":ndre_areas["poor"],
                    "area_moderate_ha":ndre_areas["moderate"],"area_good_ha":ndre_areas["good"],"area_excellent_ha":ndre_areas["excellent"]})

                # ── KPI 2: Water ───────────────────────────────────────────
                ndwi_areas = _bucket_areas(valid_ndwi, px_to_ha, _NDWI_BUCKETS)
                water_rows.append({"date":date_str,"parcel_id":parcel_id,"ndwi_mean":ndwi_mean,"ndwi_stdv":ndwi_std,
                    "area_verydry_ha":ndwi_areas["very_dry"],"area_dry_ha":ndwi_areas["dry"],
                    "area_moderate_ha":ndwi_areas["moderate"],"area_wet_ha":ndwi_areas["wet"],"area_verywet_ha":ndwi_areas["very_wet"]})

                # ── KPI 3: Harvest ─────────────────────────────────────────
                mri = _compute_MRI(dap, ndwi_mean, total_cycle)
                harvest_rows.append({"date":date_str,"parcel_id":parcel_id,"days_since_planting":dap,"ndwi_mean":ndwi_mean,"mri_score":mri})

                # ── KPI 4: Fertilizer ──────────────────────────────────────
                n_result = _compute_NDI_VRA(ndre_mean, ndwi_mean, ndre_std, row["area_ha"], dap)
                stress_map = {"SUFFICIENT":0,"WATER_STRESS_ONLY":1,"WATER_FIRST":2,"N_DEFICIENT":3}
                vra_map    = {"NO_N":0,"UNIFORM_LOW":1,"UNIFORM_HIGH":2,"VRA_NEEDED":3}
                fert_rows.append({"date":date_str,"parcel_id":parcel_id,"ndre_mean":ndre_mean,"ndre_stdv":ndre_std,"ndwi_mean":ndwi_mean,
                    "stress_type":stress_map.get(n_result["stress_type"],-1),"ndi_score":n_result["NDI"],
                    "n_rate_kg_ha":n_result["N_rate_kg_ha"],"vra_priority":vra_map.get(n_result["VRA_priority"],-1),
                    "p_rate_kg_ha":0.0,"k_rate_kg_ha":0.0,"zn_rate_kg_ha":0.0})

            except Exception as e:
                print(f"   ⚠️ Error on parcel {parcel_id}: {e}"); continue

        return pd.DataFrame(health_rows), pd.DataFrame(water_rows), pd.DataFrame(harvest_rows), pd.DataFrame(fert_rows)

    # ── Weed module (standalone, from Processing_Code_new.ipynb) ─────────
    def _run_weed_module(date_str, ndre_path, parcels_gdf, sowing_data):
        """
        Simple NDRE-based weed detection:
        - Threshold = parcel NDRE mean + 1.5*STD
        - Pixels > threshold flagged as potential weed hotspots
        - Computes weed area (ha), weed %, persistence tracking, alert level
        """
        from rasterio.features import shapes as rasterio_shapes, rasterize as rasterio_rasterize
        print(f"   🌿 Weed module: {date_str}")
        with rasterio.open(ndre_path) as src:
            raster_crs = src.crs
        parcels_utm = parcels_gdf.to_crs(raster_crs)
        weed_rows = []
        for idx, row in parcels_utm.iterrows():
            parcel_id = row["parcel_id"]
            geom = [row.geometry]
            try:
                with rasterio.open(ndre_path) as src:
                    out_ndre, out_tr = rasterio_mask(src, geom, crop=True, all_touched=False)
                    nd_val = src.nodata if src.nodata is not None else 0
                    meta = src.meta.copy()
                data = out_ndre[0]
                valid_mask = (data != nd_val)
                if np.sum(valid_mask) == 0: continue
                valid_ndre = data[valid_mask]
                ndre_mean  = float(np.mean(valid_ndre))
                ndre_std   = float(np.std(valid_ndre))
                threshold  = ndre_mean + 1.5 * ndre_std
                # Binary weed mask (1 = weed candidate, 0 = normal)
                weed_bin = np.zeros_like(data, dtype="uint8")
                weed_bin[valid_mask] = (data[valid_mask] > threshold).astype("uint8")
                total_valid_px = np.sum(valid_mask)
                px_to_ha = row["area_ha"] / total_valid_px if total_valid_px > 0 else 0
                weed_px = np.sum(weed_bin)
                weed_ha  = weed_px * px_to_ha
                weed_pct = (weed_ha / row["area_ha"] * 100) if row["area_ha"] > 0 else 0
                wpi = weed_pct / 100  # simplified WPI
                alert = 2 if weed_pct >= 10 else (1 if weed_pct >= 5 else 0)

                # Optionally save weed binary raster
                if GENERATE_WEED_MAPS:
                    meta.update({"height":data.shape[0],"width":data.shape[1],"transform":out_tr,"dtype":"uint8","nodata":0})
                    with rasterio.open(KPI_OUTPUT_DIR/f"{parcel_id}_WEED_BINARY_{date_str}.tif","w",**meta) as dst:
                        dst.write(weed_bin[np.newaxis,:,:])

                weed_rows.append({"date":date_str,"parcel_id":parcel_id,"weed_area_ha":weed_ha,
                    "weed_pct":round(weed_pct,2),"weed_pressure_index":round(wpi,4),"alert_level":alert})
            except Exception as e:
                print(f"     ⚠️ Weed error on {parcel_id}: {e}"); continue
        return pd.DataFrame(weed_rows)

    # ── Load parcels ───────────────────────────────────────────────────────
    print("\n[1/4] Loading parcel GeoJSON...")
    if not PARCELS_GEOJSON.exists():
        raise FileNotFoundError(f"Parcels GeoJSON not found: {PARCELS_GEOJSON}")
    parcels = gpd.read_file(PARCELS_GEOJSON)
    if "area_ha" not in parcels.columns:
        parcels_utm_tmp = parcels.to_crs("EPSG:32643")
        parcels["area_ha"] = parcels_utm_tmp.geometry.area / 10000
    sowing_data = _load_sowing_dates(parcels)
    print(f"   ✅ {len(parcels)} parcels loaded")

    # ── Determine date string from INFERENCE_DATE ──────────────────────────
    # Processing_Code_new.ipynb expects folders named "YYYY_MM_DD"
    _date_str = INFERENCE_DATE.replace("-","_")   # "2026_05_03"

    # ── Run per-date processing using the cloud-fill TIFFs ────────────────
    print("\n[2/4] Running KPI extraction...")
    all_health  = pd.DataFrame()
    all_water   = pd.DataFrame()
    all_harvest = pd.DataFrame()
    all_fert    = pd.DataFrame()
    all_weed    = pd.DataFrame()

    h, w, r, f = _process_date_folder(_date_str, _ndre_input, _ndwi_input, parcels, sowing_data)
    all_health  = pd.concat([all_health,  h], ignore_index=True)
    all_water   = pd.concat([all_water,   w], ignore_index=True)
    all_harvest = pd.concat([all_harvest, r], ignore_index=True)
    all_fert    = pd.concat([all_fert,    f], ignore_index=True)

    weed_df = _run_weed_module(_date_str, _ndre_input, parcels, sowing_data)
    all_weed = pd.concat([all_weed, weed_df], ignore_index=True)

# ── Updated : Save parquets ──────────────────────────────────────────────────────
    print("\n[3/4] Saving KPI parquet files...")
    _parquet_paths = {}
    _date_suffix = INFERENCE_DATE.replace("-", "_") # e.g., 2026_05_03
    
    for df, name in [
        (all_health,  "Jalna_DATA_HEALTH"),
        (all_water,   "Jalna_DATA_WATER"),
        (all_harvest, "Jalna_DATA_HARVEST"),
        (all_fert,    "Jalna_DATA_FERTILIZER"),
        (all_weed,    "Jalna_DATA_WEED"),
    ]:
        # Filename now includes the date: Jalna_DATA_HEALTH_2026_05_03.parquet
        out_path = KPI_OUTPUT_DIR / f"{name}_{_date_suffix}.parquet" 
        df.to_parquet(out_path, index=False)
        _parquet_paths[name] = out_path
        print(f"   ✅ {out_path.name} ({len(df)} rows)")

    # ── Summary ───────────────────────────────────────────────────────────
    print("\n[4/4] KPI Processing Summary:")
    print(f"   Parcels processed    : {len(parcels)}")
    print(f"   Inference date used  : {INFERENCE_DATE}")
    print(f"   Health rows          : {len(all_health)}")
    print(f"   Water rows           : {len(all_water)}")
    print(f"   Harvest rows         : {len(all_harvest)}")
    print(f"   Fertilizer rows      : {len(all_fert)}")
    print(f"   Weed rows            : {len(all_weed)}")
    print(f"\n   All outputs saved to : {KPI_OUTPUT_DIR}")
    print("\n✅ KPI PIPELINE COMPLETE")

    KPI_PARQUET_PATHS = _parquet_paths


## 🪣 Cell 5 — S3 Upload: Raw Pairs Data

Uploads the entire `pairs/` folder (S1 + S2 raw files) to S3 for long-term backup.
This preserves the original imagery so the cloud-fill model can be re-run later
without re-downloading from Copernicus.

**Destination:** `s3://<S3_BUCKET>/<S3_RAW_PREFIX>/`


In [ ]:
# ============================================================================
# CELL 5: S3 UPLOAD — RAW PAIRS DATA (pairs/ → S3)
# ============================================================================
# Source: S3_transfer.ipynb (Local → S3 section)
# Uploads all files in PAIRS_DIR recursively to the S3 raw prefix.
# ============================================================================

if not RUN_S3_UPLOAD_RAW:
    print("⏭️  RUN_S3_UPLOAD_RAW = False — skipping raw data S3 upload.")
else:
    print("="*70); print("S3 UPLOAD: RAW PAIRS DATA"); print("="*70)
    print(f"  Source      : {PAIRS_DIR}")
    print(f"  Destination : s3://{S3_BUCKET}/{S3_RAW_PREFIX}/")

    s3_client = boto3.client("s3")
    _raw_files = [f for f in Path(PAIRS_DIR).rglob("*") if f.is_file()]
    _raw_total  = sum(f.stat().st_size for f in _raw_files)

    print(f"  Files found : {len(_raw_files)}")
    print(f"  Total size  : {_raw_total/(1024**2):.2f} MB")
    print()

    from tqdm import tqdm as _tqdm

    if not _raw_files:
        print("⚠️  No files found in PAIRS_DIR — nothing to upload.")
    else:
        with _tqdm(total=_raw_total, unit="B", unit_scale=True, desc="Uploading raw pairs") as pbar:
            for _local_path in _raw_files:
                _rel      = _local_path.relative_to(PAIRS_DIR)
                _s3_key   = f"{S3_RAW_PREFIX}/{str(_rel).replace(os.sep, '/')}"
                s3_client.upload_file(
                    str(_local_path),
                    S3_BUCKET,
                    _s3_key,
                    Callback=pbar.update
                )
        print(f"\n✅ Raw pairs uploaded to s3://{S3_BUCKET}/{S3_RAW_PREFIX}/")


## 🪣 Cell 6 — S3 Upload: Processed Outputs

Uploads all cloud-fill TIFFs and KPI parquet files to S3.

**Destination:** `s3://<S3_BUCKET>/<S3_OUTPUT_PREFIX>/`

Two sub-folders are created:
- `cloudfill/` — NDRE/NDWI fused TIFFs, full S2 fused stack, tier map
- `kpi/` — Parquet files for all KPI tables


In [ ]:
# ============================================================================
# CELL 6: S3 UPLOAD — PROCESSED OUTPUTS (cloudfill + KPI → S3)
# ============================================================================
# Source: S3_transfer.ipynb
# Uploads cloud-fill TIFFs and KPI parquet files to the S3 output prefix.
# ============================================================================

if not RUN_S3_UPLOAD_OUTPUTS:
    print("⏭️  RUN_S3_UPLOAD_OUTPUTS = False — skipping outputs S3 upload.")
else:
    print("="*70); print("S3 UPLOAD: PROCESSED OUTPUTS"); print("="*70)
    print(f"  Destination : s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}/")

    s3_client = boto3.client("s3")

    from tqdm import tqdm as _tqdm

    # ── Helper: upload a list of files with a given S3 subfolder ─────────
    def _upload_file_list(file_list, s3_subfolder, label):
        """Upload a list of Paths to s3://<S3_BUCKET>/<S3_OUTPUT_PREFIX>/<subfolder>/."""
        total_sz = sum(f.stat().st_size for f in file_list if Path(f).exists())
        if not file_list or total_sz == 0:
            print(f"   ℹ️  No files to upload for {label}.")
            return
        print(f"\n  📤 {label}: {len(file_list)} files | {total_sz/(1024**2):.1f} MB")
        with _tqdm(total=total_sz, unit="B", unit_scale=True, desc=f"  Uploading {label}") as pbar:
            for lp in file_list:
                lp = Path(lp)
                if not lp.exists():
                    print(f"   ⚠️ Skipping missing file: {lp.name}"); continue
                _s3_key = f"{S3_OUTPUT_PREFIX}/{s3_subfolder}/{lp.name}"
                s3_client.upload_file(str(lp), S3_BUCKET, _s3_key, Callback=pbar.update)

    # ── Upload cloud-fill TIFFs ────────────────────────────────────────────
    _date_str = INFERENCE_DATE.replace("-", "_")

    # 1. Upload ONLY cloud-fill TIFFs for the current date
    _cf_tiffs = list(CLOUDFILL_TIFF_DIR.glob(f"*{_date_str}*.tif")) 
    _upload_file_list(_cf_tiffs, "cloudfill", f"Cloud-fill TIFFs for {_date_str}")

    # 2. Upload ONLY KPI parquets for the current date
    _kpi_parquets = list(KPI_OUTPUT_DIR.glob(f"*{_date_str}*.parquet"))
    _upload_file_list(_kpi_parquets, "kpi", f"KPI Parquets for {_date_str}")
    
    # 3. Upload the extracted parcel TIFF subfolder for the current date
    _parcel_tiffs = list((KPI_OUTPUT_DIR / _date_str).glob("*.tif")) if (KPI_OUTPUT_DIR / _date_str).exists() else []
    _upload_file_list(_parcel_tiffs, f"kpi/{_date_str}", f"Extracted Parcel TIFFs for {_date_str}")

    print(f"\n✅ Outputs uploaded to s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}/")
    print(f"   cloudfill/ — {len(_cf_tiffs)} TIF files")
    print(f"   kpi/       — {len(_kpi_parquets)} parquet files")


## 📋 Cell 7 — Pipeline Summary

In [ ]:
# ============================================================================
# CELL 7: PIPELINE SUMMARY
# ============================================================================
# Prints a full summary of everything produced during this pipeline run.
# No computation here — safe to re-run at any time.
# ============================================================================

from pathlib import Path as _P
import datetime as _dt

print("\n" + "="*76)
print("  🛰️  JALNA END-TO-END PIPELINE — COMPLETE SUMMARY")
print("="*76)
print(f"  Run timestamp      : {_dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  AOI GeoJSON        : {AOI_GEOJSON}")
print(f"  Parcels GeoJSON    : {PARCELS_GEOJSON}")
print(f"  Inference window   : {fecha_inicio.date()} → {fecha_fin.date()}")
print(f"  Inference date     : {INFERENCE_DATE}")
print(f"  Previous dates     : {PREVIOUS_DATES}")
print()
print(f"  STAGE RESULTS:")
print(f"    1. Download      : {'✅ RAN' if RUN_DOWNLOAD else '⏭️  SKIPPED'}")
print(f"    2. Cloud-fill    : {'✅ RAN' if RUN_CLOUDFILL else '⏭️  SKIPPED'}")
print(f"    3. KPI           : {'✅ RAN' if RUN_KPI else '⏭️  SKIPPED'}")
print(f"    4a. S3 (raw)     : {'✅ RAN' if RUN_S3_UPLOAD_RAW else '⏭️  SKIPPED'}")
print(f"    4b. S3 (outputs) : {'✅ RAN' if RUN_S3_UPLOAD_OUTPUTS else '⏭️  SKIPPED'}")
print()

# ── Pairs folder summary ──────────────────────────────────────────────────
print(f"  PAIRS/ FOLDER: {PAIRS_DIR}")
if PAIRS_DIR.exists():
    for folder in sorted(PAIRS_DIR.iterdir()):
        if folder.is_dir():
            _m = __import__("re").match(r"(inference|prev\d+)_\d{4}-\d{2}-\d{2}", folder.name)
            if not _m: continue
            tifs = list(folder.glob("*.tif"))
            print(f"    {folder.name}/")
            for tf in sorted(tifs):
                _sz = tf.stat().st_size/1024/1024
                try:
                    import rasterio as _ras
                    with _ras.open(tf) as src: _bands = src.count
                except: _bands = "?"
                _type = "S2(15b)" if _bands==15 else ("S1(2b)" if _bands==2 else f"?({_bands}b)")
                print(f"      └── {tf.name:<50} ({_sz:.1f} MB, {_type})")

# ── Cloud-fill outputs ────────────────────────────────────────────────────
print()
print(f"  CLOUD-FILL OUTPUTS: {CLOUDFILL_TIFF_DIR}")
if Path(CLOUDFILL_TIFF_DIR).exists():
    for tf in sorted(_P(CLOUDFILL_TIFF_DIR).glob("*.tif")):
        _sz = tf.stat().st_size/1024/1024
        print(f"    {tf.name:<65} ({_sz:.1f} MB)")
else:
    print("    (directory not found)")

# ── KPI parquets ──────────────────────────────────────────────────────────
print()
print(f"  KPI PARQUETS: {KPI_OUTPUT_DIR}")
for _pq in sorted(KPI_OUTPUT_DIR.glob("*.parquet")):
    import pandas as _pd
    try:
        _df = _pd.read_parquet(_pq)
        print(f"    {_pq.name:<45} ({len(_df)} rows, {_pq.stat().st_size/1024:.1f} KB)")
    except Exception:
        print(f"    {_pq.name} (unreadable)")

# ── S3 locations ──────────────────────────────────────────────────────────
print()
print(f"  S3 BACKUP LOCATIONS:")
if RUN_S3_UPLOAD_RAW:
    print(f"    Raw pairs : s3://{S3_BUCKET}/{S3_RAW_PREFIX}/")
if RUN_S3_UPLOAD_OUTPUTS:
    print(f"    Outputs   : s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}/cloudfill/")
    print(f"    KPI       : s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}/kpi/")
print()
print("="*76)
print("  ✅ PIPELINE FINISHED")
print("="*76)
